# Biomedical Word Representations on CORD-19

How can a machine learn what biomedical words mean? We build the full pipeline on
**CORD-19**, a corpus of scientific articles on COVID-19: we extract the text, compare four
ways of splitting it into tokens, learn word vectors with three different methods, and
check what the vectors have learnt about the vocabulary of COVID-19.

Team project, DSI Summer School 2025, Imperial College London.

**Contents**

1. Parse the data
2. Tokenization: `split()`, NLTK and SciSpaCy, a pretrained subword tokenizer, our own BPE
3. Word representations: N-gram co-occurrence + SVD, skip-gram with negative sampling, BERT
4. Exploring the representations: t-SNE, biomedical entities, co-occurrence, semantic similarity

**Scale.** Parsing and tokenization run on the whole corpus. Learning the word vectors is
far more expensive, so the representations of Part 3 are trained on samples
(1,000 documents for the N-gram vectors, 100 sentences for the skip-gram), which a laptop
GPU can handle. The results of Part 4 should be read with this in mind.

## Setup

Download the 2021-07-26 release of CORD-19 from
[Semantic Scholar](https://www.semanticscholar.org/cord19/download) and extract
`document_parses.tar.gz` into `data/`:

```sh
mkdir -p data && cd data
wget https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2021-07-26/document_parses.tar.gz
tar -xf document_parses.tar.gz        # creates data/document_parses/
```

Every intermediate file (corpus, tokenized documents, vocabularies, vectors, figures) is
written to `OUTPUT_DIR`, so each part can be run on its own once the previous one is done.

In [ ]:
import os

DATA_DIR = "data"        # contains document_parses/
OUTPUT_DIR = "output"    # corpus, tokenized files, embeddings and figures
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Parse the data

CORD-19 stores each article as a JSON file (about 425,000 of them in `pdf_json` and
`pmc_json`). We walk through every file, join the title, the abstract and the body text into
one string, and write one document per line to `full_corpus.txt`. Writing as we go keeps the
memory use low: the whole corpus is parsed in 10–15 minutes on a laptop.

In [ ]:
import os
import json
from tqdm import tqdm

# Set input directory (containing pdf_json and pmc_json) and output file path
input_dir = os.path.join(DATA_DIR, 'document_parses')
output_path = os.path.join(OUTPUT_DIR, 'full_corpus.txt')

def get_all_json_files(data_dir):
    """
    Retrieve all JSON files in subdirectories of the specified data directory.
    """
    json_files = []
    for root, _, files in os.walk(data_dir):
        for file in files:
            if file.endswith(".json"):
                json_files.append(os.path.join(root, file))
    print(f"Found {len(json_files)} JSON files in all subfolders.")
    return json_files

def extract_text(json_file):
    """
    Extract title, abstract and body text from a single JSON file.
    """
    try:
        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)
        text = []
        if "title" in data and data["title"]:
            text.append(data["title"])
        if "abstract" in data:
            text.extend([para["text"] for para in data["abstract"] if "text" in para])
        if "body_text" in data:
            text.extend([para["text"] for para in data["body_text"] if "text" in para])
        return " ".join(text).strip()
    except Exception as e:
        print(f"Failed to parse {json_file}: {e}")
        return ""

def build_corpus_streamed(input_dir, output_path):
    """
    Main workflow: stream write to the corpus file to avoid memory overflow.
    """
    all_files = get_all_json_files(input_dir)
    if not all_files:
        print("No JSON files found. Check your input path.")
        return

    with open(output_path, "w", encoding="utf-8") as f_out:
        for file in tqdm(all_files, desc="📦 Extracting documents"):
            text = extract_text(file)
            if text:
                f_out.write(text + "\n")

    print(f"\nFinished writing to: {output_path}")

if __name__ == "__main__":
    build_corpus_streamed(input_dir, output_path)

## 2. Tokenization

We apply four tokenizers to the same corpus and save, for each one, the tokenized documents
and some statistics (number of tokens, examples).

### 2.1 Python `split()`

The baseline: fast, but punctuation stays attached to the words and multi-word terms such as
"heart attack" are cut in two.

In [ ]:
import os
from tqdm import tqdm

# Function to tokenize text by simple whitespace splitting
def tokenize_split(text):
    """
    Split input text into tokens based on whitespace.
    """
    return text.split()

# File paths configuration
input_path = os.path.join(OUTPUT_DIR, 'full_corpus.txt')
token_stats_path = os.path.join(OUTPUT_DIR, 'split_token_stats.txt')
tokenized_output_path = os.path.join(OUTPUT_DIR, 'split_tokenized_docs.txt')

def process_split_tokenization(input_path, token_stats_path, tokenized_output_path):
    """
    Read the corpus file line by line, tokenize each document,
    write token counts to stats file and tokenized documents to output file.
    Also prints a few sample tokenizations at the end.
    """
    total_tokens = 0
    sample_docs = []

    # Count total number of documents for progress bar
    with open(input_path, "r", encoding="utf-8") as f:
        total_docs = sum(1 for _ in f)

    # Open input and two output files
    with open(input_path, "r", encoding="utf-8") as f_in, \
         open(token_stats_path, "w", encoding="utf-8") as f_stat, \
         open(tokenized_output_path, "w", encoding="utf-8") as f_tok:

        for idx, line in enumerate(tqdm(f_in, total=total_docs, desc="🔧 Tokenizing (split)")):
            line = line.strip()
            if not line:
                continue

            tokens = tokenize_split(line)

            # Collect first 3 sample document tokenizations
            if len(sample_docs) < 3:
                sample_docs.append((line[:100], tokens[:10]))

            # Write token count per document
            f_stat.write(f"Doc {idx + 1}: {len(tokens)} tokens\n")
            # Write tokenized text (tokens separated by ' | ')
            f_tok.write(" | ".join(tokens) + "\n")

            total_tokens += len(tokens)

    # Final summary printout
    print(f"\nDone! Processed {total_docs} documents.")
    print(f"Total tokens (split): {total_tokens}")
    print(f"Token stats file: {token_stats_path}")
    print(f"Tokenized output file: {tokenized_output_path}")

    # Print sample tokenization results
    print("\nSample tokenization:")
    for i, (text, tokens) in enumerate(sample_docs, start=1):
        print(f"\nSample Doc #{i}")
        print(f"Original snippet: {text}")
        print(f"Tokens: {tokens}")

if __name__ == "__main__":
    process_split_tokenization(input_path, token_stats_path, tokenized_output_path)

### 2.2 NLTK and SciSpaCy

NLTK is a general-purpose tokenizer. SciSpaCy (`en_core_sci_sm`) is trained on biomedical
text and keeps terms like "COVID-19" in one piece. We also remove punctuation and
meaningless symbols. The SciSpaCy output is the input of Parts 3 and 4.

In [ ]:
import os
from tqdm import tqdm
import string

import nltk
from nltk.tokenize import word_tokenize
from nltk.tokenize.punkt import PunktSentenceTokenizer
from nltk.corpus import stopwords
import spacy

# Download required NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

# Load the English Punkt sentence tokenizer
tokenizer_path = nltk.data.find("tokenizers/punkt/english.pickle")
tokenizer = PunktSentenceTokenizer(tokenizer_path.open())

# ====== Load SciSpaCy model ======
nlp = spacy.load("en_core_sci_sm")  # make sure it's installed

# ====== Filtering settings ======
# Basic punctuation set - only these meaningless symbols are removed
punctuation = set(string.punctuation)

# Additional meaningless symbols (if needed)
additional_meaningless_chars = {
    '...', '–', '—', '"', '"', "'", "'", '`', "''",
    '•', '·', '※', '→', '←', '↑', '↓'
}

# Combine all symbols to filter out
meaningless_symbols = punctuation.union(additional_meaningless_chars)

def filter_tokens(tokens):
    """
    Remove only meaningless punctuation symbols; keep all English words (including 'a', 'an', 'to', etc.)

    Args:
        tokens: list of tokens produced by a tokenizer

    Returns:
        list of filtered tokens
    """
    filtered = []
    for token in tokens:
        # Filter conditions:
        # 1. Not purely whitespace
        # 2. Not in meaningless symbols
        # 3. Not composed entirely of meaningless symbols
        if (token.strip() and  # not empty or whitespace
            token not in meaningless_symbols and  # not a meaningless symbol
            not all(c in meaningless_symbols for c in token)):  # not purely symbols
            filtered.append(token)
    return filtered

# ======= TOKENIZERS ======
def nltk_tokenizer(text):
    tokens = word_tokenize(text)
    return filter_tokens(tokens)

def scispacy_tokenizer(text):
    tokens = [token.text for token in nlp(text)]
    return filter_tokens(tokens)

# ===== Path configuration ======
input_path = os.path.join(OUTPUT_DIR, 'full_corpus.txt')

# Output paths for NLTK
nltk_stats_path = os.path.join(OUTPUT_DIR, 'nltk_token_stats.txt')
nltk_tokens_path = os.path.join(OUTPUT_DIR, 'nltk_tokenized_docs.txt')

# Output paths for SciSpaCy
sci_stats_path = os.path.join(OUTPUT_DIR, 'scispacy_token_stats.txt')
sci_tokens_path = os.path.join(OUTPUT_DIR, 'scispacy_tokenized_docs.txt')

# ===== Common processing function ======
def process_tokenizer(input_path, tokenizer, stats_out_path, tokens_out_path,
                      label="Tokenizer", max_docs=1000):
    total_raw_tokens = 0
    total_filtered_tokens = 0
    samples = []

    # Count total documents
    with open(input_path, "r", encoding="utf-8") as f:
        total_docs = sum(1 for _ in f)

    with open(input_path, "r", encoding="utf-8") as fin, \
         open(stats_out_path, "w", encoding="utf-8") as fstat, \
         open(tokens_out_path, "w", encoding="utf-8") as ftok:

        # Write report header
        fstat.write(f"=== {label} Tokenization Report ===\n")
        fstat.write("Filter condition: Remove only meaningless punctuation, keep all English words\n")
        fstat.write(f"Number of filtered punctuation symbols: {len(meaningless_symbols)}\n")
        fstat.write("=" * 50 + "\n\n")

        for idx, line in enumerate(tqdm(fin, total=min(total_docs, max_docs), desc=f"🔧 Tokenizing ({label})")):
            if idx >= max_docs:
                break

            line = line.strip()
            if not line:
                continue

            try:
                # Get raw token count for stats
                if label == "NLTK":
                    raw = word_tokenize(line)
                else:
                    raw = [token.text for token in nlp(line)]

                # Get filtered tokens
                filtered = tokenizer(line)
            except Exception as e:
                print(f"⚠️ Tokenizer error at Doc {idx + 1}: {e}")
                continue

            # Collect sample docs for display
            if len(samples) < 3:
                samples.append((line[:100], raw[:15], filtered[:15]))

            # Write per-document stats
            fstat.write(f"Doc {idx + 1}:\n")
            fstat.write(f"  Raw tokens: {len(raw)}\n")
            fstat.write(f"  Filtered tokens: {len(filtered)}\n")
            fstat.write(f"  Filter rate: {((len(raw) - len(filtered)) / len(raw) * 100):.1f}%\n\n")

            # Write filtered tokens to output
            if filtered:
                ftok.write(" ".join(filtered) + "\n")

            total_raw_tokens += len(raw)
            total_filtered_tokens += len(filtered)

    # Overall statistics
    overall_rate = ((total_raw_tokens - total_filtered_tokens) / total_raw_tokens * 100) if total_raw_tokens > 0 else 0

    print(f"\nFinished: {label}")
    print(f"Total Docs Processed: {min(total_docs, max_docs)}")
    print(f"Total Raw Tokens: {total_raw_tokens}")
    print(f"Total Filtered Tokens: {total_filtered_tokens}")
    print(f"Overall Filter Rate: {overall_rate:.1f}%")
    print(f"Stats file saved to: {stats_out_path}")
    print(f"Tokenized documents saved to: {tokens_out_path}")

    print("\nSample tokenization results:")
    for i, (text, raw, filtered) in enumerate(samples):
        print(f"\nSample Doc #{i + 1}")
        print(f"Original: {text}")
        print(f"Raw tokens: {raw}")
        print(f"Filtered tokens: {filtered}")

    # Append overall stats to stats file
    with open(stats_out_path, "a", encoding="utf-8") as fstat:
        fstat.write("=" * 50 + "\n")
        fstat.write("Overall Statistics:\n")
        fstat.write(f"Documents processed: {min(total_docs, max_docs)}\n")
        fstat.write(f"Total raw tokens: {total_raw_tokens}\n")
        fstat.write(f"Total filtered tokens: {total_filtered_tokens}\n")
        fstat.write(f"Overall filter rate: {overall_rate:.1f}%\n")
        fstat.write(f"Average raw tokens per doc: {total_raw_tokens/min(total_docs, max_docs):.1f}\n")
        fstat.write(f"Average filtered tokens per doc: {total_filtered_tokens/min(total_docs, max_docs):.1f}\n")

# ===== Main entry point =====
if __name__ == "__main__":
    print("Starting tokenization process...")
    print(f"Number of punctuation symbols to filter: {len(meaningless_symbols)}")
    print("Filter condition: Remove only meaningless punctuation, keep all English words (including 'a', 'an', 'to', etc.)")
    print("=" * 60)

    # NLTK tokenization (first 500 docs by default)
    print("\nRunning NLTK tokenizer...")
    process_tokenizer(
        input_path,
        tokenizer=nltk_tokenizer,
        stats_out_path=nltk_stats_path,
        tokens_out_path=nltk_tokens_path,
        label="NLTK",
        max_docs=500  # adjust number of docs to process
    )

    print("\n" + "="*60)

    # SciSpaCy tokenization (first 500 docs by default)
    print("\nRunning SciSpaCy tokenizer...")
    process_tokenizer(
        input_path,
        tokenizer=scispacy_tokenizer,
        stats_out_path=sci_stats_path,
        tokens_out_path=sci_tokens_path,
        label="SciSpaCy",
        max_docs=500  # adjust number of docs to process
    )

    print("\nAll tokenization tasks completed!")

### 2.3 A pretrained subword tokenizer (SciBERT)

Subword tokenizers split rare words into frequent pieces, so no word is ever unknown:
"tracheostomy" becomes `trache` + `##ostomy`. We use the vocabulary of SciBERT, trained on
scientific papers (PubMedBERT and BioBERT can be selected instead).

In [ ]:
import os
from tqdm import tqdm
from transformers import AutoTokenizer

# ====== Configuration paths ======
input_path = os.path.join(OUTPUT_DIR, 'full_corpus.txt')
bpe_stats_path = os.path.join(OUTPUT_DIR, 'bpe_token_stats.txt')
bpe_tokens_path = os.path.join(OUTPUT_DIR, 'bpe_tokenized_docs.txt')
bpe_vocab_dir = os.path.join(OUTPUT_DIR, 'bpe_vocab')

# ====== Configure the BPE tokenizer ======
BIOMEDICAL_MODELS = {
    "SciBERT": "allenai/scibert_scivocab_uncased",
    "PubMedBERT": "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
    "BioBERT": "dmis-lab/biobert-v1.1"
}
SELECTED_MODEL = "SciBERT"

# Number of documents per batch
BATCH_SIZE = 32
# Maximum documents to process
MAX_DOCS = 1000

# ====== Optimized processing function ======
def process_bpe_tokenizer(
    input_path, tokenizer, stats_out_path, tokens_out_path, vocab_dir,
    batch_size=32, max_docs=1000
):
    # Create vocab directory (if needed) and save tokenizer files
    os.makedirs(vocab_dir, exist_ok=True)
    tokenizer.save_pretrained(vocab_dir)

    total_tokens = 0
    total_docs = 0
    sample_docs = []
    medical_term_examples = []

    # First, count total lines for progress tracking
    with open(input_path, 'r', encoding='utf-8') as f:
        total_lines = sum(1 for _ in f)
    num_to_process = min(total_lines, max_docs)

    # Write header for the stats file
    with open(stats_out_path, 'w', encoding='utf-8') as f_stat:
        f_stat.write(f"Model: {SELECTED_MODEL} ({tokenizer.__class__.__name__})\n")
        f_stat.write(f"Vocab size: {tokenizer.vocab_size}\n")
        f_stat.write("Doc # | Token Count\n")
        f_stat.write("-" * 30 + "\n")

    # Open files for reading input, writing tokenized output, and appending stats
    with open(input_path, 'r', encoding='utf-8') as f_in, \
         open(tokens_out_path, 'w', encoding='utf-8') as f_tok, \
         open(stats_out_path, 'a', encoding='utf-8') as f_stat:

        batch = []
        for idx, line in enumerate(tqdm(f_in, total=num_to_process, desc="🔧 Tokenizing batches")):
            # Stop once we've processed max_docs
            if idx >= max_docs:
                break
            text = line.strip()
            if not text:
                continue

            batch.append((idx, text))
            # When batch is full or at end, process it
            if len(batch) == batch_size or idx == num_to_process - 1:
                indices, texts = zip(*batch)
                # Batch-encode without special tokens or masks
                encodings = tokenizer(
                    list(texts),
                    add_special_tokens=False,
                    return_attention_mask=False,
                    return_token_type_ids=False
                )
                for doc_idx, tokens in zip(indices, encodings.tokens()):
                    token_list = tokens
                    f_tok.write("|".join(token_list) + "\n")
                    f_stat.write(f"{doc_idx+1:6} | {len(token_list):5}\n")
                    total_tokens += len(token_list)
                    total_docs += 1

                    # Capture up to 3 sample documents
                    if len(sample_docs) < 3:
                        snippet = texts[doc_idx - indices[0]][:100]
                        sample_docs.append((snippet, token_list[:20]))
                    # Capture medical-term examples (e.g., 'covid', 'virus')
                    if (len(medical_term_examples) < 5 and
                        any(t.lower().startswith(('covid', 'virus')) for t in token_list)):
                        medical_term_examples.append((texts[doc_idx - indices[0]], token_list))

                batch = []

    # Compute averages and print summary
    avg_tokens = total_tokens / total_docs if total_docs else 0
    print(f"Processed {total_docs} docs, total tokens: {total_tokens}, avg per doc: {avg_tokens:.1f}")
    print(f"Stats saved to {stats_out_path}, tokens to {tokens_out_path}, vocab to {vocab_dir}")

    # Display sample outputs
    print("\nSample tokenization results:")
    for i, (orig, toks) in enumerate(sample_docs, 1):
        print(f"Sample #{i}: {orig}... -> {toks}")
    if medical_term_examples:
        print("\nMedical term examples:")
        for i, (orig, toks) in enumerate(medical_term_examples, 1):
            print(f"Example #{i}: tokens containing medical term -> {toks[:30]}")

if __name__ == "__main__":
    print("Initializing BPE tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(BIOMEDICAL_MODELS[SELECTED_MODEL])
    process_bpe_tokenizer(
        input_path,
        tokenizer,
        bpe_stats_path,
        bpe_tokens_path,
        bpe_vocab_dir,
        batch_size=BATCH_SIZE,
        max_docs=MAX_DOCS
    )

### 2.4 Training our own BPE tokenizer

General vocabularies do not know many medical terms, so we train a Byte-Pair Encoding
tokenizer on the corpus itself (vocabulary of 30,000 subwords, minimum frequency 2).

In [ ]:
import os
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# 1. Prepare the file paths for the biomedical corpus
# Please place all .txt files in the specified directory
corpus_file = os.path.join(OUTPUT_DIR, 'full_corpus.txt')
file_paths = [corpus_file]

if not file_paths:
    raise ValueError(f"No .txt files found, please check the directory path: {corpus_file}")

# 2. Initialize the BPE tokenizer
# Use [UNK] to represent unknown tokens
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

# 3. Set up the BPE training parameters
trainer = BpeTrainer(
    vocab_size=30000,      # Vocabulary size, can be adjusted based on corpus size
    min_frequency=2,       # Ignore subwords that appear less than 2 times
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
)

# 4. Train the tokenizer
print("Starting BPE tokenizer training...")
tokenizer.train(files=file_paths, trainer=trainer)
print(f"Training completed, vocabulary size: {tokenizer.get_vocab_size()}")

# 5. Save the model files
output_dir = OUTPUT_DIR
os.makedirs(output_dir, exist_ok=True)
tokenizer.model.save(output_dir)
tokenizer.save(os.path.join(output_dir, "tokenizer.json"))
print(f"Model saved to: {output_dir}")

# 6. Example usage
text = "Patient shows high levels of C-reactive protein and elevated liver enzymes."
encoded = tokenizer.encode(text)
print("Original text: ", text)
print("Encoded ids: ", encoded.ids)
print("Subword tokens: ", encoded.tokens)
print("Decoded text: ", tokenizer.decode(encoded.ids))

## 3. Word representations

### 3.1 N-gram co-occurrence + SVD

We keep the 20,000 most frequent tokens (frequency ≥ 3), count how often two words appear
in the same 3-gram, and compress the co-occurrence matrix to 256 dimensions with a truncated
SVD. The 256 dimensions keep 15.7% of the variance: enough to capture broad patterns, but a
lot of information is lost.

In [ ]:
import os
import numpy as np
from collections import Counter
from tqdm import tqdm
from sklearn.decomposition import TruncatedSVD

# ====== Configuration parameters ======
N = 3               # Size of each N-gram
TOP_K = 20000       # Keep only the TOP_K most frequent tokens
VECTOR_DIM = 256    # Dimensionality of word vectors
MIN_FREQ = 3        # Minimum token frequency to include
MAX_DOCS = 1000     # Maximum number of documents to process

# ====== File paths ======
TOKENIZED_FILE = os.path.join(OUTPUT_DIR, 'scispacy_tokenized_docs.txt')
VOCAB_PATH = os.path.join(OUTPUT_DIR, 'ngram_vocab.txt')
EMBEDDING_PATH = os.path.join(OUTPUT_DIR, 'ngram_vectors.npy')
WORDS_VECTORS_PATH = os.path.join(OUTPUT_DIR, 'ngram_words_vectors.txt')
STATS_PATH = os.path.join(OUTPUT_DIR, 'ngram_stats.txt')

# ====== Step 1: Build vocabulary ======
def build_vocab(file_path, top_k=TOP_K, min_freq=MIN_FREQ, max_docs=MAX_DOCS):
    """
    Count token frequencies across up to max_docs lines, filter out low-frequency tokens,
    then take the top_k most common and assign each a unique index (with special tokens).
    """
    print(f"Building vocabulary (top_k={top_k}, min_freq={min_freq})...")
    counter = Counter()
    processed = 0

    # Count frequencies
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc="Counting tokens"):
            if processed >= max_docs:
                break
            line = line.strip()
            if not line:
                continue
            tokens = line.split()  # assume whitespace-separated tokens
            counter.update(tokens)
            processed += 1

    print(f"Total unique tokens: {len(counter)}")
    print(f"Total token occurrences: {sum(counter.values())}")

    # Filter by minimum frequency and take top_k
    filtered = [(w, c) for w, c in counter.items() if c >= min_freq]
    filtered.sort(key=lambda x: x[1], reverse=True)
    filtered = filtered[:top_k] if len(filtered) > top_k else filtered
    print(f"Tokens after filtering: {len(filtered)}")

    # Reserve special tokens at the front
    special = ["<UNK>", "<s>", "</s>"]
    vocab = {tok: idx for idx, tok in enumerate(special)}
    for word, _ in filtered:
        if word not in vocab:
            vocab[word] = len(vocab)
    return vocab, counter

# ====== Step 2: Build N-gram co-occurrence matrix ======
def build_cooccurrence_matrix(file_path, vocab, n=N, max_docs=MAX_DOCS, window_size=5):
    """
    Create a (vocab_size x vocab_size) co-occurrence matrix where
    each center word (last in the N-gram) accumulates counts of its context words.
    """
    print(f"🔧 Building {n}-gram co-occurrence matrix...")
    V = len(vocab)
    cooc = np.zeros((V, V), dtype=np.float32)

    unk_id = vocab["<UNK>"]
    start_id = vocab["<s>"]
    end_id = vocab["</s>"]

    processed = 0
    total_ngrams = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc="Populating matrix"):
            if processed >= max_docs:
                break
            line = line.strip()
            if not line:
                continue
            tokens = line.split()
            # Map each token to its ID (or UNK)
            ids = [vocab.get(tok, unk_id) for tok in tokens]
            # Pad sentence boundaries
            ids = [start_id]*(n-1) + ids + [end_id]

            # Slide window to form N-grams
            for i in range(len(ids) - n + 1):
                center = ids[i + n - 1]
                context = ids[i:i + n - 1]
                for ctx in context:
                    cooc[center, ctx] += 1.0
                total_ngrams += 1

            processed += 1

    print(f"Documents processed: {processed}")
    print(f"Total N-grams counted: {total_ngrams}")
    return cooc

# ====== Step 3: Compute word vectors via SVD ======
def compute_word_vectors(cooc_matrix, vector_dim=VECTOR_DIM):
    """
    Convert raw co-occurrence counts into PMI, zero out negatives, then
    perform Truncated SVD to obtain lower-dimensional embeddings.
    """
    print(f"🔧 Computing word vectors (dim={vector_dim})...")
    # Smoothing
    cooc_matrix += 1e-6

    # PMI transformation
    row_sum = cooc_matrix.sum(axis=1, keepdims=True)
    col_sum = cooc_matrix.sum(axis=0, keepdims=True)
    total = cooc_matrix.sum()
    pmi = np.log(cooc_matrix * total / (row_sum @ col_sum))
    pmi = np.maximum(pmi, 0)

    # SVD reduction
    print("Performing SVD decomposition...")
    svd = TruncatedSVD(n_components=min(vector_dim, pmi.shape[0]-1), random_state=42)
    vectors = svd.fit_transform(pmi)
    print(f"Explained variance ratio (sum): {svd.explained_variance_ratio_.sum():.4f}")
    return vectors

# ====== Step 4: Save outputs ======
def save_results(vocab, word_vectors, word_counts):
    """Write vocabulary, embeddings, and training statistics to disk."""
    # Vocabulary file
    print(f"Saving vocabulary to: {VOCAB_PATH}")
    with open(VOCAB_PATH, 'w', encoding='utf-8') as f:
        for word, idx in sorted(vocab.items(), key=lambda x: x[1]):
            f.write(f"{word}\t{idx}\n")

    # Numpy binary embeddings
    print(f"Saving embeddings to: {EMBEDDING_PATH}")
    np.save(EMBEDDING_PATH, word_vectors)

    # Text format embeddings (for visualization)
    print(f"Saving word+vector text to: {WORDS_VECTORS_PATH}")
    with open(WORDS_VECTORS_PATH, 'w', encoding='utf-8') as f:
        # Header: num_tokens × dim
        f.write(f"{len(vocab)} {word_vectors.shape[1]}\n")
        for word, idx in sorted(vocab.items(), key=lambda x: x[1]):
            vec_str = " ".join(f"{val:.6f}" for val in word_vectors[idx])
            f.write(f"{word} {vec_str}\n")

    # Training statistics
    print(f"Saving stats to: {STATS_PATH}")
    with open(STATS_PATH, 'w', encoding='utf-8') as f:
        f.write("=== N-gram Embedding Training Stats ===\n")
        f.write(f"N-gram size: {N}\n")
        f.write(f"Vocabulary size: {len(vocab)}\n")
        f.write(f"Vector dimension: {word_vectors.shape[1]}\n")
        f.write(f"Min token frequency: {MIN_FREQ}\n")
        f.write(f"Documents processed: {MAX_DOCS}\n\n")
        f.write("=== Top 20 Token Frequencies ===\n")
        # Sort by frequency, excluding special tokens
        freqs = [(w, word_counts.get(w, 0))
                 for w in vocab if w not in ("<UNK>", "<s>", "</s>")]
        freqs.sort(key=lambda x: x[1], reverse=True)
        for i, (w, cnt) in enumerate(freqs[:20], 1):
            f.write(f"{i:2d}. {w:<15} {cnt:>8d}\n")

# ====== Main execution ======
if __name__ == "__main__":
    print("Starting N-gram word-vector training...")

    # Step 1: Build vocab
    vocab, word_counts = build_vocab(TOKENIZED_FILE)

    # Step 2: Build co-occurrence matrix
    cooc_matrix = build_cooccurrence_matrix(TOKENIZED_FILE, vocab)

    # Step 3: Compute embeddings
    word_vectors = compute_word_vectors(cooc_matrix)

    # Step 4: Write outputs
    save_results(vocab, word_vectors, word_counts)

    print("\nTraining complete!")
    print(f"Vocabulary: {VOCAB_PATH}")
    print(f"Embeddings (npy): {EMBEDDING_PATH}")
    print(f"Embeddings (txt): {WORDS_VECTORS_PATH}")
    print(f"Stats: {STATS_PATH}")

    # Example vectors for a few common words
    print("\nSample word vectors:")
    sample_words = ["the", "a", "of", "to", "and"]
    for w in sample_words:
        if w in vocab:
            idx = vocab[w]
            vec = word_vectors[idx]
            print(f"{w}: [{vec[0]:.4f}, {vec[1]:.4f}, {vec[2]:.4f}, ...]")

### 3.2 Skip-gram with negative sampling

A small neural network learns to predict the words around each word (window of 3), against
5 random "negative" words. Vocabulary of 8,000 words, vectors of 256 dimensions, 15 epochs,
trained on an NVIDIA RTX 4060 laptop GPU on a sample of 100 sentences (349,592 tokens).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter, defaultdict
import random
from torch.utils.data import Dataset, DataLoader
import pickle
import os
from tqdm import tqdm
import time

def check_cuda_and_setup():
    """Check CUDA availability and configure device"""
    print("Checking system environment...")
    print(f"PyTorch version: {torch.__version__}")
    
    if torch.cuda.is_available():
        print("CUDA is available")
        print(f"CUDA version: {torch.version.cuda}")
        print(f"Number of GPU devices: {torch.cuda.device_count()}")
        
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            print(f"GPU {i}: {props.name}")
            print(f"Total memory: {props.total_memory / 1024**3:.1f} GB")
        
        device = 'cuda'
        print("Training will use GPU")
    else:
        print("CUDA is not available; training will use CPU")
        print("Please verify that PyTorch is installed with CUDA support")
        device = 'cpu'
    
    return device

class SkipGramDataset(Dataset):
    def __init__(self, data, word2idx, idx2word, window_size=3, negative_samples=5):
        self.data = data
        self.word2idx = word2idx
        self.idx2word = idx2word
        self.window_size = window_size
        self.negative_samples = negative_samples
        self.vocab_size = len(word2idx)
        
        # Generate training pairs
        print("Generating training samples...")
        self.samples = self._create_samples()
        
        # Build frequency distribution for negative sampling
        print("Building word frequency distribution...")
        self.word_freq = self._create_word_frequency()
        self.noise_dist = self._create_noise_distribution()
        
    def _create_samples(self):
        """Generate all positive (center, context) pairs"""
        samples = []
        for sentence in self.data:
            if len(sentence) < 2:
                continue
            for i, center_word in enumerate(sentence):
                if center_word not in self.word2idx:
                    continue
                center_idx = self.word2idx[center_word]
                start = max(0, i - self.window_size)
                end = min(len(sentence), i + self.window_size + 1)
                for j in range(start, end):
                    if j != i and sentence[j] in self.word2idx:
                        context_idx = self.word2idx[sentence[j]]
                        samples.append((center_idx, context_idx))
        return samples
    
    def _create_word_frequency(self):
        """Count word frequencies in the dataset"""
        word_count = Counter()
        for sentence in self.data:
            for word in sentence:
                if word in self.word2idx:
                    word_count[word] += 1
        freq = np.zeros(self.vocab_size)
        for word, count in word_count.items():
            freq[self.word2idx[word]] = count
        return freq
    
    def _create_noise_distribution(self):
        """Create noise distribution P(w) = U(w)^(3/4) / Z for negative sampling"""
        freq_power = np.power(self.word_freq + 1e-8, 0.75)
        return freq_power / np.sum(freq_power)
    
    def _negative_sampling(self, positive_idx):
        """Sample negatives for a given positive example index"""
        negatives = []
        while len(negatives) < self.negative_samples:
            neg = np.random.choice(self.vocab_size, p=self.noise_dist)
            if neg != positive_idx:
                negatives.append(neg)
        return negatives
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        center_idx, context_idx = self.samples[idx]
        negative_idxs = self._negative_sampling(context_idx)
        return {
            'center': torch.tensor(center_idx, dtype=torch.long),
            'context': torch.tensor(context_idx, dtype=torch.long),
            'negatives': torch.tensor(negative_idxs, dtype=torch.long)
        }

class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        
        # Embedding for center words
        self.center_embeddings = nn.Embedding(vocab_size, embedding_dim)
        # Embedding for context words
        self.context_embeddings = nn.Embedding(vocab_size, embedding_dim)
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize embedding weights uniformly"""
        bound = 0.5 / self.embedding_dim
        nn.init.uniform_(self.center_embeddings.weight, -bound, bound)
        nn.init.uniform_(self.context_embeddings.weight, -bound, bound)
    
    def forward(self, center_words, context_words, negative_words):
        """
        center_words: [batch_size] indices of center words
        context_words: [batch_size] indices of positive context words
        negative_words: [batch_size, negative_samples] indices of negative samples
        """
        center_embeds = self.center_embeddings(center_words)                            # [B, D]
        context_embeds = self.context_embeddings(context_words)                         # [B, D]
        pos_score = torch.sum(center_embeds * context_embeds, dim=1)                    # [B]
        pos_loss = -torch.log(torch.sigmoid(pos_score + 1e-8))                          # [B]
        
        neg_embeds = self.context_embeddings(negative_words)                            # [B, neg, D]
        center_exp = center_embeds.unsqueeze(1)                                         # [B, 1, D]
        neg_score = torch.sum(center_exp * neg_embeds, dim=2)                           # [B, neg]
        neg_loss = -torch.sum(torch.log(torch.sigmoid(-neg_score + 1e-8)), dim=1)       # [B]
        
        loss = torch.mean(pos_loss + neg_loss)
        return loss
    
    def get_embeddings(self):
        """Return learned center embeddings as numpy array"""
        return self.center_embeddings.weight.data.cpu().numpy()

class SkipGramTrainer:
    def __init__(self, model, dataset, learning_rate=0.003, device='cuda'):
        self.device = device
        self.model = model.to(device)
        self.dataset = dataset
        self.optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-5)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, patience=2, factor=0.5)
        
    def train(self, epochs=10, batch_size=512, save_path=None, patience=3):
        """Train the Skip-gram model with negative sampling"""
        if self.device == 'cuda':
            dataloader = DataLoader(
                self.dataset, batch_size=batch_size, shuffle=True,
                num_workers=0, pin_memory=True
            )
        else:
            dataloader = DataLoader(
                self.dataset, batch_size=min(batch_size, 512),
                shuffle=True, num_workers=2
            )
        
        self.model.train()
        best_loss = float('inf')
        patience_counter = 0
        
        print(f"Starting training on device: {self.device}")
        if self.device == 'cuda':
            try:
                print(f"Using GPU: {torch.cuda.get_device_name()}")
                print(f"Initial GPU memory: {torch.cuda.memory_allocated() / 1024**2:.1f} MB")
            except Exception as e:
                print(f"Failed to query GPU info: {e}")
        
        print(f"Number of training samples: {len(self.dataset):,}")
        print(f"Batch size: {batch_size}")
        print(f"Number of batches: {len(dataloader):,}")
        print("-" * 60)
        
        for epoch in range(epochs):
            epoch_loss = 0.0
            start_time = time.time()
            bar = tqdm(dataloader, desc=f'Epoch {epoch+1}/{epochs}')
            
            for batch_idx, batch in enumerate(bar):
                center = batch['center'].to(self.device, non_blocking=True)
                context = batch['context'].to(self.device, non_blocking=True)
                negatives = batch['negatives'].to(self.device, non_blocking=True)
                
                self.optimizer.zero_grad()
                loss = self.model(center, context, negatives)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
                
                epoch_loss += loss.item()
                
                if batch_idx % 50 == 0:
                    lr = self.optimizer.param_groups[0]['lr']
                    info = {'Loss': f'{loss.item():.4f}', 'LR': f'{lr:.6f}'}
                    if self.device == 'cuda':
                        gpu_mem = torch.cuda.memory_allocated() / 1024**2
                        info['GPU_MB'] = f'{gpu_mem:.0f}'
                    bar.set_postfix(info)
                
                if self.device == 'cuda' and batch_idx % 500 == 0:
                    torch.cuda.empty_cache()
            
            avg_loss = epoch_loss / len(dataloader)
            elapsed = time.time() - start_time
            self.scheduler.step(avg_loss)
            
            print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Time: {elapsed:.1f}s")
            
            if avg_loss < best_loss:
                best_loss = avg_loss
                patience_counter = 0
                if save_path:
                    best_path = f"{save_path}_best.pth"
                    torch.save({
                        'model_state_dict': self.model.state_dict(),
                        'optimizer_state_dict': self.optimizer.state_dict(),
                        'epoch': epoch,
                        'loss': avg_loss
                    }, best_path)
                    print(f"Best model saved to: {best_path}")
            else:
                patience_counter += 1
            
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}, best loss: {best_loss:.4f}")
                break
            
            if self.device == 'cuda':
                torch.cuda.empty_cache()
        
        return self.model

def load_and_preprocess_data(file_path, min_count=3, max_vocab_size=8000, max_sentences=300):
    """Load text data and preprocess into tokenized sentences and vocab mappings"""
    print(f"Loading data from: {file_path}")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    
    sentences, total_words = [], 0
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if line:
                words = line.split()
                if len(words) > 1:
                    sentences.append(words)
                    total_words += len(words)
            if len(sentences) >= max_sentences:
                break
            if (i + 1) % 500 == 0:
                print(f"Processed {i+1} lines, collected {len(sentences)} valid sentences")
    
    print(f"Loaded {len(sentences)} sentences, total words: {total_words:,}")
    
    word_count = Counter()
    for sent in sentences:
        word_count.update(sent)
    print(f"Original vocabulary size: {len(word_count):,}")
    
    filtered = {w: c for w, c in word_count.items() if c >= min_count}
    print(f"Filtered vocabulary size: {len(filtered):,} (min_count >= {min_count})")
    
    if len(filtered) > max_vocab_size:
        most_common = Counter(filtered).most_common(max_vocab_size)
        vocab_words = {w for w, _ in most_common}
        print(f"Limiting vocabulary to top {len(vocab_words):,}")
    else:
        vocab_words = set(filtered.keys())
    
    word2idx = {w: i for i, w in enumerate(sorted(vocab_words))}
    idx2word = {i: w for w, i in word2idx.items()}
    
    filtered_sentences = []
    for sent in sentences:
        filtered_sent = [w for w in sent if w in word2idx]
        if len(filtered_sent) > 1:
            filtered_sentences.append(filtered_sent)
    
    print(f"Final number of sentences: {len(filtered_sentences)}")
    print(f"Final vocabulary size: {len(vocab_words)}")
    return filtered_sentences, word2idx, idx2word

def test_similarity(embeddings, word2idx, idx2word, test_words=None, top_k=5):
    """Evaluate word similarity using cosine similarity"""
    def find_most_similar(word, top_k):
        if word not in word2idx:
            return f"Word '{word}' not in vocabulary"
        idx = word2idx[word]
        vec = embeddings[idx]
        sims = np.dot(embeddings, vec) / (
            np.linalg.norm(embeddings, axis=1) * np.linalg.norm(vec) + 1e-8
        )
        top_idxs = np.argsort(sims)[::-1][1:top_k+1]
        return [(idx2word[i], sims[i]) for i in top_idxs]
    
    if test_words is None:
        all_w = list(word2idx.keys())
        test_words = random.sample(all_w, min(8, len(all_w)))
    
    print("\n" + "="*60)
    print("Word Embedding Similarity Test")
    print("="*60)
    for w in test_words:
        if w in word2idx:
            sim = find_most_similar(w, top_k)
            print(f"\nWords most similar to '{w}':")
            for i, (sw, score) in enumerate(sim, 1):
                print(f"   {i}. {sw:<20} (score: {score:.4f})")

def main():
    """Main function — optimized for CUDA 11.6"""
    device = check_cuda_and_setup()
    
    DATA_PATH = os.path.join(OUTPUT_DIR, 'scispacy_tokenized_docs.txt')
    EMBEDDING_DIM = 256
    WINDOW_SIZE = 3
    NEGATIVE_SAMPLES = 5
    LEARNING_RATE = 0.003
    EPOCHS = 15
    MIN_COUNT = 3
    MAX_VOCAB_SIZE = 8000
    MAX_SENTENCES = 100
    
    if device == 'cuda':
        BATCH_SIZE = 2056  # Larger batch for GPU
        print("GPU-optimized configuration")
    else:
        BATCH_SIZE = 512   # Smaller batch for CPU
        EPOCHS = 10        # Fewer epochs on CPU
        print("CPU-optimized configuration")
    
    SKIPGRAM_DIR = os.path.join(OUTPUT_DIR, 'skipgram')
    os.makedirs(SKIPGRAM_DIR, exist_ok=True)
    
    try:
        print("\nSkip-gram with Negative Sampling Training")
        print("="*60)
        
        # Step 1: Data preprocessing
        print("Step 1: Data preprocessing")
        sentences, word2idx, idx2word = load_and_preprocess_data(
            DATA_PATH, min_count=MIN_COUNT,
            max_vocab_size=MAX_VOCAB_SIZE,
            max_sentences=MAX_SENTENCES
        )
        vocab_size = len(word2idx)
        
        # Save vocabulary mappings
        vocab_file = os.path.join(SKIPGRAM_DIR, "vocabulary.pkl")
        with open(vocab_file, 'wb') as f:
            pickle.dump({'word2idx': word2idx, 'idx2word': idx2word}, f)
        print(f"Vocabulary saved to: {vocab_file}")
        
        # Step 2: Build dataset
        print("\nStep 2: Building training dataset")
        dataset = SkipGramDataset(
            sentences, word2idx, idx2word,
            window_size=WINDOW_SIZE,
            negative_samples=NEGATIVE_SAMPLES
        )
        print(f"Number of training samples: {len(dataset):,}")
        
        # Step 3: Initialize model
        print("\nStep 3: Initializing Skip-gram model")
        model = SkipGramModel(vocab_size, EMBEDDING_DIM)
        total_params = sum(p.numel() for p in model.parameters())
        print(f"Model parameter count: {total_params:,}")
        print(f"Embedding dimension: {EMBEDDING_DIM}")
        print(f"Vocabulary size: {vocab_size}")
        
        # Step 4: Train
        print("\nStep 4: Training")
        trainer = SkipGramTrainer(model, dataset, LEARNING_RATE, device)
        save_base = os.path.join(SKIPGRAM_DIR, "skipgram_model")
        trained_model = trainer.train(
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            save_path=save_base,
            patience=4
        )
        
        # Step 5: Save outputs
        print("\nStep 5: Saving training outputs")
        embeddings = trained_model.get_embeddings()
        emb_path = os.path.join(SKIPGRAM_DIR, "word_embeddings.npy")
        np.save(emb_path, embeddings)
        print(f"Embeddings saved to: {emb_path}")
        print(f"Embedding shape: {embeddings.shape}")
        
        txt_path = os.path.join(SKIPGRAM_DIR, "embeddings.txt")
        with open(txt_path, 'w', encoding='utf-8') as fw:
            for idx, vec in enumerate(embeddings):
                fw.write(idx2word[idx] + " " + " ".join(f"{v:.6f}" for v in vec) + "\n")
        print(f"Embeddings text file saved to: {txt_path}")
        
        # Save final model checkpoint
        final_model_file = os.path.join(SKIPGRAM_DIR, "final_skipgram_model.pth")
        torch.save({
            'model_state_dict': trained_model.state_dict(),
            'vocab_size': vocab_size,
            'embedding_dim': EMBEDDING_DIM,
            'word2idx': word2idx,
            'idx2word': idx2word,
            'training_config': {
                'window_size': WINDOW_SIZE,
                'negative_samples': NEGATIVE_SAMPLES,
                'min_count': MIN_COUNT,
                'max_sentences': MAX_SENTENCES,
                'learning_rate': LEARNING_RATE,
                'device': device
            }
        }, final_model_file)
        print(f"Complete model saved to: {final_model_file}")
        
        # Save training config text
        config_file = os.path.join(SKIPGRAM_DIR, "training_config.txt")
        with open(config_file, 'w', encoding='utf-8') as f:
            f.write("Skip-gram Training Configuration\n")
            f.write("===============================\n")
            f.write(f"Embedding dimension: {EMBEDDING_DIM}\n")
            f.write(f"Window size: {WINDOW_SIZE}\n")
            f.write(f"Negative samples: {NEGATIVE_SAMPLES}\n")
            f.write(f"Learning rate: {LEARNING_RATE}\n")
            f.write(f"Batch size: {BATCH_SIZE}\n")
            f.write(f"Epochs: {EPOCHS}\n")
            f.write(f"Vocabulary size: {vocab_size}\n")
            f.write(f"Training samples: {len(dataset):,}\n")
            f.write(f"Device: {device}\n")
        
        # Step 6: Evaluate similarity
        test_similarity(embeddings, word2idx, idx2word)
        
        print("\n" + "="*60)
        print("Training complete!")
        print("="*60)
        print(f"Output directory: {SKIPGRAM_DIR}")
        print(f"Embedding dimension: {EMBEDDING_DIM}")
        print(f"Vocabulary size: {vocab_size:,}")
        print(f"Training samples: {len(dataset):,}")
        print(f"Epochs: {EPOCHS}")
        print(f"Device used: {device}")
        
        avg_norm = np.mean(np.linalg.norm(embeddings, axis=1))
        print(f"Average embedding norm: {avg_norm:.4f}")
        
        if device == 'cuda':
            print(f"Final GPU memory allocated: {torch.cuda.memory_allocated() / 1024**2:.1f} MB")
            print(f"Peak GPU memory allocated: {torch.cuda.max_memory_allocated() / 1024**2:.1f} MB")
        
    except Exception as e:
        print(f"An error occurred during training: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if device == 'cuda':
            torch.cuda.empty_cache()

if __name__ == "__main__":
    main()

### 3.3 Contextual representations with BERT

The two methods above give one fixed vector per word. A masked language model like BERT
gives a different vector depending on the sentence. We compare the vectors of "stick" in two
sentences where it has different meanings, and let BERT fill in a masked word.

In [ ]:
import torch
from transformers import BertTokenizer, BertModel, BertForMaskedLM

# Initialize tokenizer and model
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
mlm = BertForMaskedLM.from_pretrained(model_name)

# Function to get contextual embeddings for a target word in a sentence

def get_contextual_embedding(sentence, target_word, device='cpu'):
    model.to(device)

    # Tokenize and get input IDs
    encoded = tokenizer(sentence, return_tensors='pt')
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)

    # Forward pass through BERT
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        # outputs.last_hidden_state shape: [batch_size, seq_len, hidden_size]
        hidden_states = outputs.last_hidden_state.squeeze(0)

    # Find the token indices corresponding to target_word
    tokens = tokenizer.tokenize(sentence)
    word_pieces = tokenizer.tokenize(target_word)
    # locate the first sub-token of the target
    for i in range(len(tokens) - len(word_pieces) + 1):
        if tokens[i:i+len(word_pieces)] == word_pieces:
            start, end = i+1, i+1+len(word_pieces)  # +1 for [CLS]
            # Average embeddings if wordpiece-split
            embedding = hidden_states[start:end].mean(dim=0)
            return embedding
    raise ValueError(f"'{target_word}' not found in tokenized sentence.")

# Example sentences with polysemous word 'stick'
sent1 = "He used a stick to play hockey on the ice."
sent2 = "The label won\'t stick to the wet surface."

def compare_embeddings(word, sent1, sent2):
    emb1 = get_contextual_embedding(sent1, word)
    emb2 = get_contextual_embedding(sent2, word)
    # Cosine similarity
    cos = torch.nn.functional.cosine_similarity(emb1.unsqueeze(0), emb2.unsqueeze(0)).item()
    return cos

if __name__ == '__main__':
    word = 'stick'
    sim = compare_embeddings(word, sent1, sent2)
    print(f"Cosine similarity between 'stick' in different contexts: {sim:.4f}")

    # Masked Language Modeling: predict masked tokens
    masked_sentence = "He used a [MASK] to play hockey on the ice."
    inputs = tokenizer(masked_sentence, return_tensors='pt')
    mask_idx = (inputs['input_ids'][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0].item()

    with torch.no_grad():
        outputs = mlm(**inputs)
        logits = outputs.logits
    softmax = torch.nn.functional.softmax(logits, dim=-1)
    mask_scores = softmax[0, mask_idx]
    topk = torch.topk(mask_scores, 5)

    print("Top predictions for [MASK]:")
    for score, idx in zip(topk.values, topk.indices):
        print(tokenizer.decode(idx), f"({score:.4f})")

## 4. Exploring the representations

### 4.1 t-SNE of 1,000 random words

t-SNE projects the 256-dimensional skip-gram vectors on a plane, and k-means (k = 8) colours
the groups. Numbers, years and German words form their own clusters, but most clusters have
no clear meaning.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from collections import Counter
import random
from pathlib import Path
import warnings
import torch
import pickle  # Added for loading .pkl files

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')

class RandomWordVisualizer:
    def __init__(self, vectors_path, vocab_path=None):
        """
        Initialize the visualizer
        Args:
            vectors_path: Path to word vectors file (.txt, .npy, .pth/.pt)
            vocab_path: Path to vocabulary file (.txt, .npy, .pkl), used for npy/pth or custom vocab scenarios
        """
        self.vectors_path = Path(vectors_path)
        self.vocab_path = Path(vocab_path) if vocab_path else None
        self.vectors = None
        self.vocab = None
        self.vector_dim = None

    def load_word_vectors(self):
        """Load word vectors and vocabulary based on file extension
        Supports .txt (first line: size dim), .npy (vectors only), .pth/.pt (dict containing 'vectors' and 'vocab')"""
        ext = self.vectors_path.suffix.lower()
        print(f"Loading word vectors file [{self.vectors_path.name}]...")
        try:
            if ext == '.txt':
                with open(self.vectors_path, 'r', encoding='utf-8') as f:
                    # Read header to get vocab size and vector dimension
                    first_line = f.readline().strip().split()
                    vocab_size, self.vector_dim = map(int, first_line[:2])
                    vectors_list, vocab_list = [], []
                    # Keep only words with letters, numbers, or Chinese characters
                    pattern = re.compile(r'^[\u4e00-\u9fa5A-Za-z0-9]+$')
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) != self.vector_dim + 1:
                            continue
                        word = parts[0]
                        if not pattern.match(word):
                            continue
                        try:
                            vector = [float(x) for x in parts[1:]]
                        except ValueError:
                            continue
                        if len(vector) != self.vector_dim:
                            continue
                        vocab_list.append(word)
                        vectors_list.append(vector)
                self.vocab = vocab_list
                self.vectors = np.array(vectors_list)

            elif ext == '.npy':
                self.vectors = np.load(self.vectors_path, allow_pickle=False)
                self.vector_dim = self.vectors.shape[1]
                if not self.vocab_path:
                    raise ValueError("vocab_path is required when using .npy vectors")
                vext = self.vocab_path.suffix.lower()
                if vext == '.npy':
                    self.vocab = np.load(self.vocab_path, allow_pickle=True).tolist()
                elif vext == '.pkl':
                    with open(self.vocab_path, 'rb') as vf:
                        self.vocab = pickle.load(vf)
                else:
                    with open(self.vocab_path, 'r', encoding='utf-8') as vf:
                        self.vocab = [w.strip() for w in vf if w.strip()]

            elif ext in ['.pth', '.pt']:
                data = torch.load(self.vectors_path, map_location='cpu')
                if isinstance(data, dict) and 'vectors' in data:
                    vecs = data['vectors']
                    self.vectors = vecs.cpu().numpy() if isinstance(vecs, torch.Tensor) else np.array(vecs)
                    self.vector_dim = self.vectors.shape[1]
                else:
                    raise ValueError("'vectors' key not found in .pth file, please check format")
                if 'vocab' in data:
                    self.vocab = data['vocab']
                elif self.vocab_path:
                    vext = self.vocab_path.suffix.lower()
                    if vext == '.npy':
                        self.vocab = np.load(self.vocab_path, allow_pickle=True).tolist()
                    elif vext == '.pkl':
                        with open(self.vocab_path, 'rb') as vf:
                            self.vocab = pickle.load(vf)
                    else:
                        with open(self.vocab_path, 'r', encoding='utf-8') as vf:
                            self.vocab = [w.strip() for w in vf if w.strip()]
                else:
                    raise ValueError("'vocab' key not found in .pth file and no vocab_path provided")

            else:
                raise ValueError(f"Unsupported file type: {ext}")

            if len(self.vocab) != len(self.vectors):
                print(f"Warning: vocabulary size ({len(self.vocab)}) does not match vector count ({len(self.vectors)})")
            print(f"Loaded: {len(self.vocab)} words, {self.vector_dim}-dim vectors")
            return True
        except Exception as e:
            print(f"Load failed: {e}")
            return False

    # Other methods remain unchanged

    def random_sampling(self, n_samples=1000):
        """Randomly sample words"""
        print(f"Randomly sampling {n_samples} words...")
        if len(self.vocab) <= n_samples:
            print(f"Not enough words, using all {len(self.vocab)} words")
            return self.vectors, self.vocab, list(range(len(self.vocab)))
        random.seed(42)
        np.random.seed(42)
        all_indices = list(range(len(self.vocab)))
        selected_indices = random.sample(all_indices, n_samples)
        selected_indices = sorted(selected_indices)
        sampled_vectors = self.vectors[selected_indices]
        sampled_vocab = [self.vocab[i] for i in selected_indices]
        print(f"Sample result:")
        print(f"  - Total words: {len(sampled_vocab)}")
        print(f"  - Sample ratio: {len(sampled_vocab)/len(self.vocab)*100:.1f}%")
        print(f"  - Index range: {min(selected_indices)} - {max(selected_indices)}")
        return sampled_vectors, sampled_vocab, selected_indices

    def run_clustering(self, vectors, vocab, n_clusters=8):
        """Perform K-means clustering"""
        print(f"Running K-means clustering (k={n_clusters})...")
        scaler = StandardScaler()
        vectors_scaled = scaler.fit_transform(vectors)
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        labels = kmeans.fit_predict(vectors_scaled)
        cluster_info = []
        for i in range(n_clusters):
            cluster_words = [vocab[j] for j in range(len(vocab)) if labels[j] == i]
            cluster_info.append({
                'cluster_id': i,
                'total_words': len(cluster_words),
                'sample_words': cluster_words[:10]
            })
            print(f"  Cluster {i+1}: {len(cluster_words)} words")
            print(f"    Sample words: {', '.join(cluster_words[:5])}")
        return labels, cluster_info

    def select_random_labels(self, vocab, embeddings, cluster_labels, n_labels=100):
        """Randomly select words to label"""
        print(f"Randomly selecting {n_labels} label words...")
        random.seed(42)
        np.random.seed(42)
        all_indices = list(range(len(vocab)))
        n_labels = min(n_labels, len(all_indices))
        selected_indices = random.sample(all_indices, n_labels)
        print(f"Label selection result:")
        print(f"  - Number of labels: {len(selected_indices)}")
        print(f"  - Label ratio: {len(selected_indices)/len(vocab)*100:.1f}%")
        return selected_indices

    def create_2d_plot(self, embeddings, vocab, cluster_labels=None, save_path=None):
        """Generate 2D visualization"""
        print("Generating 2D visualization...")
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 12))
        label_indices = self.select_random_labels(vocab, embeddings, cluster_labels, 100)
        if cluster_labels is not None:
            colors = plt.cm.Set3(np.linspace(0, 1, len(np.unique(cluster_labels))))
            for cluster_id in np.unique(cluster_labels):
                mask = cluster_labels == cluster_id
                ax1.scatter(embeddings[mask, 0], embeddings[mask, 1],
                            c=[colors[cluster_id]], label=f'Cluster {cluster_id+1}',
                            alpha=0.7, s=50)
        for i in label_indices:
            word = vocab[i]
            x, y = embeddings[i, 0], embeddings[i, 1]
            bbox = dict(boxstyle="round,pad=0.2", facecolor='lightblue',
                        alpha=0.8, edgecolor='blue')
            ax1.annotate(word, (x, y), fontsize=8, fontweight='normal',
                         color='black', bbox=bbox, alpha=0.9)
        ax1.set_title(f'Cluster View - Randomly Selected 100 Labels\n(Total {len(vocab)} Words)',
                      fontsize=16, fontweight='bold', pad=20)
        ax1.set_xlabel('t-SNE Dimension 1', fontsize=12)
        ax1.set_ylabel('t-SNE Dimension 2', fontsize=12)
        ax1.grid(True, alpha=0.3)
        if cluster_labels is not None:
            ax1.legend(loc='upper right', fontsize=10)
        scatter = ax2.scatter(embeddings[:, 0], embeddings[:, 1],
                              c=range(len(vocab)), cmap='viridis',
                              alpha=0.6, s=30)
        sample_labels = random.sample(label_indices, min(50, len(label_indices)))
        for i in sample_labels:
            word = vocab[i]
            x, y = embeddings[i, 0], embeddings[i, 1]
            ax2.annotate(word, (x, y), fontsize=8, fontweight='normal',
                         color='black', alpha=0.8,
                         bbox=dict(boxstyle="round,pad=0.2", facecolor='white',
                                   alpha=0.8, edgecolor='gray'))
        ax2.set_title(f'Word Distribution View\n(Total {len(vocab)} Random Words)',
                      fontsize=16, fontweight='bold', pad=20)
        ax2.set_xlabel('t-SNE Dimension 1', fontsize=12)
        ax2.set_ylabel('t-SNE Dimension 2', fontsize=12)
        ax2.grid(True, alpha=0.3)
        plt.colorbar(scatter, ax=ax2, label='Word Index')
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
            print(f"2D plot saved: {save_path}")
        plt.show()
        return fig

    def create_3d_plot(self, embeddings, vocab, cluster_labels=None, save_path=None):
        """Generate 3D visualization"""
        print("Generating 3D visualization...")
        fig = plt.figure(figsize=(20, 15))
        ax = fig.add_subplot(111, projection='3d')
        label_indices = self.select_random_labels(vocab, embeddings, cluster_labels, 80)
        if cluster_labels is not None:
            colors = plt.cm.Set3(np.linspace(0, 1, len(np.unique(cluster_labels))))
            for cluster_id in np.unique(cluster_labels):
                mask = cluster_labels == cluster_id
                ax.scatter(embeddings[mask, 0], embeddings[mask, 1],
                           embeddings[mask, 2], c=[colors[cluster_id]],
                           label=f'Cluster {cluster_id+1}', alpha=0.7, s=60)
        else:
            ax.scatter(embeddings[:, 0], embeddings[:, 1], embeddings[:, 2],
                       c=range(len(vocab)), cmap='viridis', alpha=0.6, s=40)
        for i in label_indices:
            word = vocab[i]
            x, y, z = embeddings[i, 0], embeddings[i, 1], embeddings[i, 2]
            ax.text(x, y, z, word, fontsize=8, fontweight='normal',
                    color='black', alpha=0.9)
        ax.set_title(f'3D t-SNE Visualization\n{len(vocab)} Random Words, {len(label_indices)} Labels',
                    fontsize=16, fontweight='bold', pad=20)
        ax.set_xlabel('t-SNE Dimension 1', fontsize=12)
        ax.set_ylabel('t-SNE Dimension 2', fontsize=12)
        ax.set_zlabel('t-SNE Dimension 3', fontsize=12)
        if cluster_labels is not None:
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
            print(f"3D plot saved: {save_path}")
        plt.show()
        return fig

    def run_tsne(self, vectors, dim=2, perplexity=30):
        """Run t-SNE dimensionality reduction"""
        print(f"Running {dim}D t-SNE...")
        scaler = StandardScaler()
        vectors_scaled = scaler.fit_transform(vectors)
        if vectors.shape[1] > 100:
            pca = PCA(n_components=50, random_state=42)
            vectors_scaled = pca.fit_transform(vectors_scaled)
            print(f"  PCA preprocessing: {vectors.shape[1]} -> 50 dimensions")
        n_samples = len(vectors)
        perplexity = min(perplexity, (n_samples - 1) // 3)
        tsne = TSNE(
            n_components=dim,
            perplexity=perplexity,
            learning_rate=max(200, n_samples // 12),
            n_iter=1000,
            random_state=42,
            init='pca'
        )
        embeddings = tsne.fit_transform(vectors_scaled)
        print(f"{dim}D t-SNE completed")
        return embeddings

    def save_detailed_results(self, vocab, embeddings_2d, embeddings_3d,
                              cluster_labels, cluster_info, output_dir):
        """Save detailed results to files"""
        print("Saving detailed results...")
        all_data = []
        for i, word in enumerate(vocab):
            all_data.append({
                'word': word,
                'x_2d': embeddings_2d[i, 0],
                'y_2d': embeddings_2d[i, 1],
                'x_3d': embeddings_3d[i, 0],
                'y_3d': embeddings_3d[i, 1],
                'z_3d': embeddings_3d[i, 2],
                'cluster': cluster_labels[i] if cluster_labels is not None else -1,
                'position_rank': i
            })
        df_all = pd.DataFrame(all_data)
        df_all.to_csv(output_dir / 'all_words_coordinates.csv', index=False, encoding='utf-8')
        with open(output_dir / 'cluster_analysis_report.txt', 'w', encoding='utf-8') as f:
            f.write("=== Word Cluster Analysis Report ===\n\n")
            f.write(f"Total words: {len(vocab)}\n")
            f.write(f"Number of clusters: {len(cluster_info)}\n\n")
            for info in cluster_info:
                f.write(f"Cluster {info['cluster_id'] + 1}:\n")
                f.write(f"  Total words: {info['total_words']}\n")
                f.write(f"  Sample words: {', '.join(info['sample_words'])}\n\n")
        with open(output_dir / 'vocabulary_stats.txt', 'w', encoding='utf-8') as f:
            f.write("=== Vocabulary Statistics Report ===\n\n")
            f.write(f"Total words: {len(vocab)}\n")
            f.write(f"Average word length: {np.mean([len(word) for word in vocab]):.2f} characters\n")
            f.write(f"Shortest word: '{min(vocab, key=len)}' ({len(min(vocab, key=len))} characters)\n")
            f.write(f"Longest word: '{max(vocab, key=len)}' ({len(max(vocab, key=len))} characters)\n\n")
            word_lengths = [len(word) for word in vocab]
            length_counter = Counter(word_lengths)
            f.write("Word length distribution:\n")
            for length, count in sorted(length_counter.items()):
                f.write(f"  {length} chars: {count} words ({count/len(vocab)*100:.1f}%)\n")
        print(f"Results saved to: {output_dir}")
        print(f"  - all_words_coordinates.csv: {len(df_all)} word coordinates")
        print(f"  - cluster_analysis_report.txt: cluster analysis report")
        print(f"  - vocabulary_stats.txt: vocabulary statistics report")
        return df_all

    def run_complete_pipeline(self, n_samples=1000, output_dir='./random_visualization_results'):
        """Execute the full visualization pipeline"""
        print("Starting random word visualization pipeline...")
        print("=" * 60)
        output_dir = Path(output_dir)
        output_dir.mkdir(exist_ok=True)
        random.seed(42)
        np.random.seed(42)
        if not self.load_word_vectors():
            print("Data loading failed, exiting...")
            return False
        vectors, vocab, indices = self.random_sampling(n_samples)
        cluster_labels, cluster_info = self.run_clustering(vectors, vocab, n_clusters=8)
        embeddings_2d = self.run_tsne(vectors, dim=2, perplexity=30)
        fig_2d = self.create_2d_plot(
            embeddings_2d, vocab, cluster_labels,
            save_path=output_dir / 'random_tsne_2d.png'
        )
        embeddings_3d = self.run_tsne(vectors, dim=3, perplexity=30)
        fig_3d = self.create_3d_plot(
            embeddings_3d, vocab, cluster_labels,
            save_path=output_dir / 'random_tsne_3d.png'
        )
        df_all = self.save_detailed_results(
            vocab, embeddings_2d, embeddings_3d, cluster_labels, cluster_info, output_dir
        )
        self.generate_summary_report(vocab, cluster_info, output_dir)
        print("\n" + "=" * 60)
        print("Random word visualization complete!")
        print(f"Output directory: {output_dir}")
        print(f"Visualized words: {len(vocab)} random words")
        print("Generated files:")
        print("  random_tsne_2d.png - 2D random word visualization")
        print("  random_tsne_3d.png - 3D random word visualization")
        print("  all_words_coordinates.csv - all word coordinates")
        print("  cluster_analysis_report.txt - cluster analysis report")
        print("  vocabulary_stats.txt - vocabulary statistics report")
        print("  visualization_summary.txt - visualization summary")
        return True

    def generate_summary_report(self, vocab, cluster_info, output_dir):
        """Generate a summary report of visualizations"""
        with open(output_dir / 'visualization_summary.txt', 'w', encoding='utf-8') as f:
            f.write("=== Random Word t-SNE Visualization Summary ===\n\n")
            f.write("Basic Statistics:\n")
            f.write(f"  - Total words visualized: {len(vocab)}\n")
            f.write(f"  - Number of clusters: {len(cluster_info)}\n")
            f.write("  - Sampling method: Random sampling\n")
            f.write("  - Label selection: Randomly select 100 words for labels\n\n")
            f.write("Cluster Analysis:\n")
            for i, info in enumerate(cluster_info):
                f.write(f"  Cluster {i+1}: {info['total_words']} words\n")
                f.write(f"    Sample: {', '.join(info['sample_words'][:5])}\n")
            f.write("\nisualization Notes:\n")
            f.write("  - 2D plot: Left shows clusters, right shows word distribution density\n")
            f.write("  - 3D plot: Shows word distribution in three dimensions\n")
            f.write("  - Words selected randomly without thematic bias\n")
            f.write("  - Label words randomly chosen for semantic insight\n\n")
            f.write("Key Insights:\n")
            f.write("  1. Natural clustering patterns of random words in semantic space\n")
            f.write("  2. Effectiveness of t-SNE for semantic similarity visualization\n")
            f.write("  3. Potential semantic groups identified by clustering algorithm\n")
            f.write("  4. Overall distribution characteristics in word vector space\n")

# Main function
def main():
    """Main - Random word visualization"""
    words_vectors_path = os.path.join(OUTPUT_DIR, 'skipgram', 'embeddings.txt')
    print("Random Word t-SNE Visualization System")
    print("=" * 50)
    print(f"Input file: {words_vectors_path}")
    visualizer = RandomWordVisualizer(words_vectors_path)
    success = visualizer.run_complete_pipeline(
        n_samples=1000,
        output_dir=os.path.join(OUTPUT_DIR, 'random_visualization_results')
    )
    if success:
        print("\nRandom word visualization task completed!")
        print("Generated high-quality 2D and 3D plots")
        print("Used random sampling of 1000 words")
        print("Randomly selected 100 words for labels")
        print("Provided detailed analysis reports and data files")
    else:
        print("\nVisualization task failed")

if __name__ == "__main__":
    main()

### 4.2 t-SNE of biomedical entities

The same projection with 1,000 words of which 40% are medical terms, sorted into categories
(anatomy, pathology, treatments, ...) so that each category has its own colour.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from collections import Counter
import random
from pathlib import Path
import warnings
import torch
import pickle  # Added for loading .pkl files

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')

class MedicalWordVisualizer:
    def __init__(self, vectors_path, vocab_path=None):
        """
        Initialize the visualizer
        Args:
            vectors_path: Path to word vectors file (.txt, .npy, .pth/.pt)
            vocab_path: Path to vocabulary file (.txt, .npy, .pkl), used for npy/pth or custom vocab scenarios
        """
        self.vectors_path = Path(vectors_path)
        self.vocab_path = Path(vocab_path) if vocab_path else None
        self.vectors = None
        self.vocab = None
        self.vector_dim = None
        
        # Initialize medical vocabulary and stopwords
        self.medical_terms = self.get_comprehensive_medical_terms()
        self.medical_affixes = self.get_medical_affixes()
        self.stopwords = self.get_stopwords()
        
        # Define medical category colors
        self.medical_colors = {
            'basic_biomedical': '#2E8B57',      # Sea Green
            'molecular_biology': '#4169E1',     # Royal Blue
            'disease_related': '#DC143C',       # Crimson
            'immune_system': '#FF6347',         # Tomato
            'treatment_drugs': '#32CD32',       # Lime Green
            'clinical_medicine': '#FF8C00',     # Dark Orange
            'research_related': '#9370DB',      # Medium Purple
            'anatomy': '#20B2AA',               # Light Sea Green
            'specific_diseases': '#B22222',     # Fire Brick
            'biochemistry': '#DAA520',          # Goldenrod
            'pharmacology': '#8A2BE2',          # Blue Violet
            'medical_technology': '#5F9EA0',    # Cadet Blue
            'professional_terms': '#CD853F'     # Peru
        }

    def get_stopwords(self):
        """Get comprehensive stopwords list"""
        return {
            'a', 'an', 'the', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with',
            'by', 'as', 'be', 'is', 'are', 'was', 'were', 'been', 'being', 'have', 'has', 'had',
            'do', 'does', 'did', 'will', 'would', 'could', 'should', 'may', 'might', 'must',
            'shall', 'can', 'this', 'that', 'these', 'those', 'i', 'you', 'he', 'she', 'it',
            'we', 'they', 'me', 'him', 'her', 'us', 'them', 'my', 'your', 'his', 'her', 'its',
            'our', 'their', 'mine', 'yours', 'hers', 'ours', 'theirs', 'myself', 'yourself',
            'himself', 'herself', 'itself', 'ourselves', 'yourselves', 'themselves', 'what',
            'which', 'who', 'whom', 'whose', 'where', 'when', 'why', 'how', 'all', 'any',
            'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor',
            'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 'just', 'now',
            'here', 'there', 'then', 'also', 'well', 'up', 'down', 'out', 'off', 'over',
            'under', 'again', 'further', 'once', 'during', 'before', 'after', 'above',
            'below', 'between', 'through', 'into', 'from', 'about', 'against', 'because',
            'until', 'while', 'although', 'if', 'unless', 'since', 'however', 'therefore',
            'thus', 'moreover', 'furthermore', 'nevertheless', 'nonetheless', 'meanwhile',
            'otherwise', 'finally', 'eventually', 'subsequently', 'accordingly',
            # Numbers and single letters
            '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
            'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm',
            'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z'
        }

    def get_comprehensive_medical_terms(self):
        """Get comprehensive medical terms list organized by categories"""
        return {
            'basic_biomedical': {
                'cell', 'cells', 'cellular', 'tissue', 'tissues', 'organ', 'organs', 'blood', 'plasma',
                'serum', 'lymph', 'heart', 'cardiac', 'lung', 'pulmonary', 'brain', 'neural', 'nerve',
                'liver', 'hepatic', 'kidney', 'renal', 'muscle', 'skeletal', 'bone', 'osseous'
            },
            'molecular_biology': {
                'protein', 'proteins', 'enzyme', 'enzymes', 'hormone', 'hormones', 'gene', 'genes',
                'genetic', 'dna', 'rna', 'mrna', 'chromosome', 'genome', 'genomic', 'nucleotide',
                'amino', 'peptide', 'polypeptide', 'receptor', 'receptors', 'antibody', 'antibodies',
                'antigen', 'antigens', 'immunoglobulin', 'cytokine', 'cytokines', 'chemokine'
            },
            'disease_related': {
                'disease', 'diseases', 'disorder', 'disorders', 'syndrome', 'syndromes', 'cancer',
                'tumor', 'tumors', 'malignant', 'benign', 'carcinoma', 'sarcoma', 'lymphoma',
                'leukemia', 'metastasis', 'metastatic', 'oncology', 'oncological', 'neoplasm',
                'infection', 'infections', 'infectious', 'virus', 'viral', 'bacteria', 'bacterial',
                'pathogen', 'pathogens', 'microorganism', 'fungal', 'parasitic', 'sepsis'
            },
            'immune_system': {
                'immune', 'immunity', 'immunology', 'immunological', 'inflammation', 'inflammatory',
                'autoimmune', 'immunodeficiency', 'allergen', 'allergic', 'allergy', 'hypersensitivity',
                'complement', 'phagocytosis', 'lymphocyte', 'macrophage', 'neutrophil', 'eosinophil'
            },
            'treatment_drugs': {
                'therapy', 'therapies', 'therapeutic', 'treatment', 'treatments', 'drug', 'drugs',
                'medication', 'pharmaceutical', 'pharmacology', 'pharmacological', 'antibiotic',
                'antiviral', 'chemotherapy', 'radiotherapy', 'immunotherapy', 'surgery', 'surgical',
                'operation', 'procedure', 'intervention', 'transplant', 'transplantation'
            },
            'clinical_medicine': {
                'clinical', 'patient', 'patients', 'diagnosis', 'diagnostic', 'symptom', 'symptoms',
                'syndrome', 'prognosis', 'prognostic', 'epidemiology', 'prevalence', 'incidence',
                'morbidity', 'mortality', 'survival', 'outcome', 'outcomes', 'healthcare', 'medical',
                'medicine', 'physician', 'doctor', 'nurse', 'hospital', 'clinic', 'laboratory'
            },
            'research_related': {
                'study', 'studies', 'research', 'trial', 'trials', 'experiment', 'experimental',
                'analysis', 'statistical', 'data', 'result', 'results', 'finding', 'findings',
                'evidence', 'cohort', 'control', 'randomized', 'placebo', 'double-blind'
            },
            'anatomy': {
                'anatomy', 'anatomical', 'physiological', 'physiology', 'pathology', 'pathological',
                'histology', 'histological', 'cytology', 'morphology', 'structure', 'function'
            },
            'specific_diseases': {
                'diabetes', 'diabetic', 'hypertension', 'cardiovascular', 'stroke', 'myocardial',
                'pneumonia', 'asthma', 'copd', 'alzheimer', 'parkinson', 'epilepsy', 'schizophrenia',
                'depression', 'anxiety', 'autism', 'arthritis', 'osteoporosis', 'obesity'
            },
            'biochemistry': {
                'biochemical', 'metabolism', 'metabolic', 'glucose', 'insulin', 'cholesterol',
                'triglyceride', 'lipid', 'carbohydrate', 'fatty', 'acid', 'vitamin', 'mineral',
                'electrolyte', 'sodium', 'potassium', 'calcium', 'magnesium', 'phosphate'
            },
            'pharmacology': {
                'dosage', 'dose', 'administration', 'absorption', 'distribution', 'excretion',
                'bioavailability', 'pharmacokinetics', 'pharmacodynamics', 'toxicity', 'adverse',
                'side-effect', 'contraindication', 'interaction'
            },
            'medical_technology': {
                'imaging', 'radiological', 'ultrasound', 'endoscopy', 'biopsy', 'culture',
                'assay', 'biomarker', 'screening', 'monitoring', 'surveillance'
            },
            'professional_terms': {
                'ibrutinib', 'cytotoxic', 'apoptosis', 'proliferation', 'differentiation',
                'angiogenesis', 'vasculature', 'endothelial', 'epithelial', 'mesenchymal',
                'stem-cell', 'regenerative', 'bioengineering', 'nanotechnology', 'personalized'
            }
        }

    def get_medical_affixes(self):
        """Get medical affixes"""
        return {
            'suffixes': [
                'osis', 'itis', 'emia', 'uria', 'pathy' 
                'ectomy', 'otomy', 'ostomy', 'scopy', 
                'phobic'
            ],
            'prefixes': [
                'anti', 'hyper', 'hypo', 'micro', 'macro', 'poly',
                'uni', 'tri', 'neo', 
                'sub', 'super', 'allo', 'homo', 'hetero', 'pseudo',
                'proto', 'meta',  'peri', 'epi', 'endo', 'exo', 'cardio',
                'neuro', 'gastro', 'hepato', 'nephro', 'pneumo', 'hemato', 'osteo'
            ]
        }

    def load_word_vectors(self):
        """Load word vectors and vocabulary, filtering out stopwords"""
        ext = self.vectors_path.suffix.lower()
        print(f"Loading word vectors file [{self.vectors_path.name}]...")
        try:
            if ext == '.txt':
                with open(self.vectors_path, 'r', encoding='utf-8') as f:
                    # Read header to get vocab size and vector dimension
                    first_line = f.readline().strip().split()
                    vocab_size, self.vector_dim = map(int, first_line[:2])
                    vectors_list, vocab_list = [], []
                    # Keep only words with letters, numbers, or Chinese characters
                    pattern = re.compile(r'^[\u4e00-\u9fa5A-Za-z0-9]+$')
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) != self.vector_dim + 1:
                            continue
                        word = parts[0]
                        if not pattern.match(word) or word.lower() in self.stopwords:
                            continue
                        try:
                            vector = [float(x) for x in parts[1:]]
                        except ValueError:
                            continue
                        if len(vector) != self.vector_dim:
                            continue
                        vocab_list.append(word)
                        vectors_list.append(vector)
                self.vocab = vocab_list
                self.vectors = np.array(vectors_list)

            # ... (other file format handling remains the same)
            elif ext == '.npy':
                self.vectors = np.load(self.vectors_path, allow_pickle=False)
                self.vector_dim = self.vectors.shape[1]
                if not self.vocab_path:
                    raise ValueError("vocab_path is required when using .npy vectors")
                vext = self.vocab_path.suffix.lower()
                if vext == '.npy':
                    vocab_temp = np.load(self.vocab_path, allow_pickle=True).tolist()
                elif vext == '.pkl':
                    with open(self.vocab_path, 'rb') as vf:
                        vocab_temp = pickle.load(vf)
                else:
                    with open(self.vocab_path, 'r', encoding='utf-8') as vf:
                        vocab_temp = [w.strip() for w in vf if w.strip()]
                
                # Filter out stopwords
                filtered_indices = [i for i, word in enumerate(vocab_temp) if word.lower() not in self.stopwords]
                self.vocab = [vocab_temp[i] for i in filtered_indices]
                self.vectors = self.vectors[filtered_indices]

            elif ext in ['.pth', '.pt']:
                data = torch.load(self.vectors_path, map_location='cpu')
                if isinstance(data, dict) and 'vectors' in data:
                    vecs = data['vectors']
                    vectors_temp = vecs.cpu().numpy() if isinstance(vecs, torch.Tensor) else np.array(vecs)
                    self.vector_dim = vectors_temp.shape[1]
                else:
                    raise ValueError("'vectors' key not found in .pth file, please check format")
                
                if 'vocab' in data:
                    vocab_temp = data['vocab']
                elif self.vocab_path:
                    vext = self.vocab_path.suffix.lower()
                    if vext == '.npy':
                        vocab_temp = np.load(self.vocab_path, allow_pickle=True).tolist()
                    elif vext == '.pkl':
                        with open(self.vocab_path, 'rb') as vf:
                            vocab_temp = pickle.load(vf)
                    else:
                        with open(self.vocab_path, 'r', encoding='utf-8') as vf:
                            vocab_temp = [w.strip() for w in vf if w.strip()]
                else:
                    raise ValueError("'vocab' key not found in .pth file and no vocab_path provided")
                
                # Filter out stopwords
                filtered_indices = [i for i, word in enumerate(vocab_temp) if word.lower() not in self.stopwords]
                self.vocab = [vocab_temp[i] for i in filtered_indices]
                self.vectors = vectors_temp[filtered_indices]

            else:
                raise ValueError(f"Unsupported file type: {ext}")

            if len(self.vocab) != len(self.vectors):
                print(f"Warning: vocabulary size ({len(self.vocab)}) does not match vector count ({len(self.vectors)})")
            print(f"Loaded: {len(self.vocab)} words, {self.vector_dim}-dim vectors (stopwords filtered)")
            return True
        except Exception as e:
            print(f"Load failed: {e}")
            return False

    def get_medical_category(self, word):
        """Determine which medical category a word belongs to"""
        word_lower = word.lower()
        
        # Check each category
        for category, terms in self.medical_terms.items():
            if word_lower in terms or word in terms:
                return category
        
        # Check for affix patterns
        for suffix in self.medical_affixes['suffixes']:
            if word_lower.endswith(suffix):
                return 'medical_affixes'
        
        for prefix in self.medical_affixes['prefixes']:
            if word_lower.startswith(prefix):
                return 'medical_affixes'
        
        return None

    def is_medical_word(self, word):
        """Check if a word is medical-related"""
        return self.get_medical_category(word) is not None

    def random_sampling(self, n_samples=1000):
        """Randomly sample words"""
        print(f"Randomly sampling {n_samples} words...")
        if len(self.vocab) <= n_samples:
            print(f"Not enough words, using all {len(self.vocab)} words")
            return self.vectors, self.vocab, list(range(len(self.vocab)))
        random.seed(42)
        np.random.seed(42)
        all_indices = list(range(len(self.vocab)))
        selected_indices = random.sample(all_indices, n_samples)
        selected_indices = sorted(selected_indices)
        sampled_vectors = self.vectors[selected_indices]
        sampled_vocab = [self.vocab[i] for i in selected_indices]
        print(f"Sample result:")
        print(f"  - Total words: {len(sampled_vocab)}")
        print(f"  - Sample ratio: {len(sampled_vocab)/len(self.vocab)*100:.1f}%")
        print(f"  - Index range: {min(selected_indices)} - {max(selected_indices)}")
        return sampled_vectors, sampled_vocab, selected_indices

    def run_clustering(self, vectors, vocab, n_clusters=8):
        """Perform K-means clustering"""
        print(f"Running K-means clustering (k={n_clusters})...")
        scaler = StandardScaler()
        vectors_scaled = scaler.fit_transform(vectors)
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        labels = kmeans.fit_predict(vectors_scaled)
        cluster_info = []
        for i in range(n_clusters):
            cluster_words = [vocab[j] for j in range(len(vocab)) if labels[j] == i]
            cluster_info.append({
                'cluster_id': i,
                'total_words': len(cluster_words),
                'sample_words': cluster_words[:10]
            })
            print(f"  Cluster {i+1}: {len(cluster_words)} words")
            print(f"    Sample words: {', '.join(cluster_words[:5])}")
        return labels, cluster_info

    def select_medical_labels(self, vocab, embeddings, cluster_labels, n_labels=100):
        """Select medical-related words for labeling, organized by category"""
        print(f"Selecting medical-related words for labeling...")
        
        # Organize medical words by category
        medical_by_category = {}
        for i, word in enumerate(vocab):
            category = self.get_medical_category(word)
            if category:
                if category not in medical_by_category:
                    medical_by_category[category] = []
                medical_by_category[category].append(i)
        
        total_medical = sum(len(indices) for indices in medical_by_category.values())
        print(f"Found {total_medical} medical-related words in {len(medical_by_category)} categories")
        
        if total_medical == 0:
            print("No medical words found, falling back to random selection")
            random.seed(42)
            all_indices = list(range(len(vocab)))
            n_labels = min(n_labels, len(all_indices))
            selected_indices = random.sample(all_indices, n_labels)
            return selected_indices, {}
        
        # Calculate proportional selection from each category
        selected_indices = []
        category_selections = {}
        
        random.seed(42)
        
        # Distribute labels proportionally across categories
        for category, indices in medical_by_category.items():
            category_ratio = len(indices) / total_medical
            category_n_labels = max(1, int(n_labels * category_ratio))
            category_n_labels = min(category_n_labels, len(indices))
            
            category_selected = random.sample(indices, category_n_labels)
            selected_indices.extend(category_selected)
            category_selections[category] = category_selected
        
        # If we have too many, randomly remove some
        if len(selected_indices) > n_labels:
            selected_indices = random.sample(selected_indices, n_labels)
            # Update category_selections accordingly
            for category in category_selections:
                category_selections[category] = [idx for idx in category_selections[category] if idx in selected_indices]
        
        print(f"Medical label selection result:")
        print(f"  - Number of labels: {len(selected_indices)}")
        print(f"  - Medical words ratio: {len(selected_indices)/len(vocab)*100:.1f}%")
        
        # Show category distribution
        for category, indices in category_selections.items():
            if indices:
                sample_words = [vocab[i] for i in indices[:3]]
                print(f"  - {category}: {len(indices)} words (e.g., {', '.join(sample_words)})")
        
        return selected_indices, category_selections

    def create_2d_plot(self, embeddings, vocab, cluster_labels=None, save_path=None):
        """Generate 2D visualization with medical word labels colored by category"""
        print("Generating 2D visualization with categorized medical labels...")
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 12))
        
        # Use medical word selection for labels
        label_indices, category_selections = self.select_medical_labels(vocab, embeddings, cluster_labels, 100)
        
        # Plot 1: Cluster view
        if cluster_labels is not None:
            colors = plt.cm.Set3(np.linspace(0, 1, len(np.unique(cluster_labels))))
            for cluster_id in np.unique(cluster_labels):
                mask = cluster_labels == cluster_id
                ax1.scatter(embeddings[mask, 0], embeddings[mask, 1],
                            c=[colors[cluster_id]], label=f'Cluster {cluster_id+1}',
                            alpha=0.7, s=50)
        
        # Add medical word labels with category colors
        for category, indices in category_selections.items():
            color = self.medical_colors.get(category, '#FF69B4')  # Default to hot pink
            for i in indices:
                word = vocab[i]
                x, y = embeddings[i, 0], embeddings[i, 1]
                bbox = dict(boxstyle="round,pad=0.2", facecolor=color,
                            alpha=0.7, edgecolor='black', linewidth=0.5)
                ax1.annotate(word, (x, y), fontsize=8, fontweight='bold',
                             color='white', bbox=bbox, alpha=0.9)
        
        ax1.set_title(f'Medical Word Cluster View - {len(label_indices)} Medical Labels\n(Total {len(vocab)} Words)',
                      fontsize=16, fontweight='bold', pad=20)
        ax1.set_xlabel('t-SNE Dimension 1', fontsize=12)
        ax1.set_ylabel('t-SNE Dimension 2', fontsize=12)
        ax1.grid(True, alpha=0.3)
        if cluster_labels is not None:
            ax1.legend(loc='upper right', fontsize=10)
        
        # Plot 2: Distribution view
        scatter = ax2.scatter(embeddings[:, 0], embeddings[:, 1],
                              c=range(len(vocab)), cmap='viridis',
                              alpha=0.6, s=30)
        
        # Sample fewer labels for the second plot to avoid overcrowding
        sample_labels = random.sample(label_indices, min(50, len(label_indices)))
        for i in sample_labels:
            word = vocab[i]
            category = self.get_medical_category(word)
            color = self.medical_colors.get(category, '#FF69B4') if category else '#FF69B4'
            x, y = embeddings[i, 0], embeddings[i, 1]
            ax2.annotate(word, (x, y), fontsize=8, fontweight='bold',
                         color='white', alpha=0.9,
                         bbox=dict(boxstyle="round,pad=0.2", facecolor=color,
                                   alpha=0.7, edgecolor='black', linewidth=0.5))
        
        ax2.set_title(f'Medical Word Distribution View\n(Total {len(vocab)} Words, {len(sample_labels)} Medical Labels)',
                      fontsize=16, fontweight='bold', pad=20)
        ax2.set_xlabel('t-SNE Dimension 1', fontsize=12)
        ax2.set_ylabel('t-SNE Dimension 2', fontsize=12)
        ax2.grid(True, alpha=0.3)
        plt.colorbar(scatter, ax=ax2, label='Word Index')
        
        # Add legend for medical categories
        legend_elements = []
        for category, color in self.medical_colors.items():
            if category in category_selections and category_selections[category]:
                legend_elements.append(plt.Rectangle((0,0),1,1, facecolor=color, alpha=0.7, label=category.replace('_', ' ').title()))
        
        if legend_elements:
            ax1.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(0, 1), fontsize=8)
        
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
            print(f"2D plot saved: {save_path}")
        plt.show()
        return fig

    def create_3d_plot(self, embeddings, vocab, cluster_labels=None, save_path=None):
        """Generate 3D visualization with medical word labels colored by category"""
        print("Generating 3D visualization with categorized medical labels...")
        fig = plt.figure(figsize=(20, 15))
        ax = fig.add_subplot(111, projection='3d')
        
        # Use medical word selection for labels
        label_indices, category_selections = self.select_medical_labels(vocab, embeddings, cluster_labels, 80)
        
        if cluster_labels is not None:
            colors = plt.cm.Set3(np.linspace(0, 1, len(np.unique(cluster_labels))))
            for cluster_id in np.unique(cluster_labels):
                mask = cluster_labels == cluster_id
                ax.scatter(embeddings[mask, 0], embeddings[mask, 1],
                           embeddings[mask, 2], c=[colors[cluster_id]],
                           label=f'Cluster {cluster_id+1}', alpha=0.7, s=60)
        else:
            ax.scatter(embeddings[:, 0], embeddings[:, 1], embeddings[:, 2],
                       c=range(len(vocab)), cmap='viridis', alpha=0.6, s=40)
        
        # Add medical word labels with category colors
        for category, indices in category_selections.items():
            color = self.medical_colors.get(category, '#FF69B4')
            for i in indices:
                word = vocab[i]
                x, y, z = embeddings[i, 0], embeddings[i, 1], embeddings[i, 2]
                ax.text(x, y, z, word, fontsize=8, fontweight='bold',
                        color=color, alpha=0.9)
        
        ax.set_title(f'3D Medical Word t-SNE Visualization\n{len(vocab)} Words, {len(label_indices)} Medical Labels',
                    fontsize=16, fontweight='bold', pad=20)
        ax.set_xlabel('t-SNE Dimension 1', fontsize=12)
        ax.set_ylabel('t-SNE Dimension 2', fontsize=12)
        ax.set_zlabel('t-SNE Dimension 3', fontsize=12)
        
        if cluster_labels is not None:
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
            print(f"3D plot saved: {save_path}")
        plt.show()
        return fig

    def run_tsne(self, vectors, dim=2, perplexity=30):
        """Run t-SNE dimensionality reduction"""
        print(f"Running {dim}D t-SNE...")
        scaler = StandardScaler()
        vectors_scaled = scaler.fit_transform(vectors)
        if vectors.shape[1] > 100:
            pca = PCA(n_components=50, random_state=42)
            vectors_scaled = pca.fit_transform(vectors_scaled)
            print(f"  PCA preprocessing: {vectors.shape[1]} -> 50 dimensions")
            n_samples = len(vectors)
        perplexity = min(perplexity, (n_samples - 1) // 3)
        tsne = TSNE(
            n_components=dim,
            perplexity=perplexity,
            learning_rate=max(200, n_samples // 12),
            n_iter=1000,
            random_state=42,
            init='pca'
        )
        embeddings = tsne.fit_transform(vectors_scaled)
        print(f"{dim}D t-SNE completed")
        return embeddings

    def save_detailed_results(self, vocab, embeddings_2d, embeddings_3d,
                              cluster_labels, cluster_info, output_dir):
        """Save detailed results to files"""
        print("Saving detailed results...")
        
        # Identify medical words and their categories for the results
        medical_labels = []
        medical_categories = []
        for i, word in enumerate(vocab):
            category = self.get_medical_category(word)
            medical_labels.append(category is not None)
            medical_categories.append(category if category else 'non_medical')
        
        all_data = []
        for i, word in enumerate(vocab):
            all_data.append({
                'word': word,
                'is_medical': medical_labels[i],
                'medical_category': medical_categories[i],
                'x_2d': embeddings_2d[i, 0],
                'y_2d': embeddings_2d[i, 1],
                'x_3d': embeddings_3d[i, 0],
                'y_3d': embeddings_3d[i, 1],
                'z_3d': embeddings_3d[i, 2],
                'cluster': cluster_labels[i] if cluster_labels is not None else -1,
                'position_rank': i
            })
        
        df_all = pd.DataFrame(all_data)
        df_all.to_csv(output_dir / 'all_words_coordinates.csv', index=False, encoding='utf-8')
        
        # Save medical words separately
        df_medical = df_all[df_all['is_medical'] == True]
        df_medical.to_csv(output_dir / 'medical_words_coordinates.csv', index=False, encoding='utf-8')
        
        # Generate cluster analysis report
        with open(output_dir / 'cluster_analysis_report.txt', 'w', encoding='utf-8') as f:
            f.write("=== Medical Word Cluster Analysis Report ===\n\n")
            f.write(f"Total words: {len(vocab)}\n")
            f.write(f"Medical words: {sum(medical_labels)} ({sum(medical_labels)/len(vocab)*100:.1f}%)\n")
            f.write(f"Number of clusters: {len(cluster_info)}\n\n")
            
            # Category distribution
            category_counts = Counter(medical_categories)
            f.write("Medical Category Distribution:\n")
            for category, count in category_counts.items():
                if category != 'non_medical':
                    f.write(f"  - {category.replace('_', ' ').title()}: {count} words ({count/len(vocab)*100:.1f}%)\n")
            f.write(f"  - Non-medical words: {category_counts.get('non_medical', 0)} words\n\n")
            
            # Cluster details
            for info in cluster_info:
                cluster_words = [vocab[j] for j in range(len(vocab)) if cluster_labels[j] == info['cluster_id']]
                medical_in_cluster = [(word, self.get_medical_category(word)) for word in cluster_words if self.is_medical_word(word)]
                f.write(f"Cluster {info['cluster_id'] + 1}:\n")
                f.write(f"  Total words: {info['total_words']}\n")
                f.write(f"  Medical words: {len(medical_in_cluster)}\n")
                f.write(f"  Sample words: {', '.join(info['sample_words'])}\n")
                if medical_in_cluster:
                    medical_examples = [word for word, cat in medical_in_cluster[:5]]
                    f.write(f"  Medical examples: {', '.join(medical_examples)}\n")
                    # Show category breakdown in cluster
                    cluster_categories = Counter([cat for word, cat in medical_in_cluster])
                    f.write(f"  Medical categories: {dict(cluster_categories)}\n")
                f.write("\n")
        
        # Generate medical vocabulary statistics
        with open(output_dir / 'medical_vocabulary_stats.txt', 'w', encoding='utf-8') as f:
            medical_words = [word for word, is_med in zip(vocab, medical_labels) if is_med]
            f.write("=== Medical Vocabulary Statistics Report ===\n\n")
            f.write(f"Total words: {len(vocab)}\n")
            f.write(f"Medical words: {len(medical_words)} ({len(medical_words)/len(vocab)*100:.1f}%)\n")
            f.write(f"Non-medical words: {len(vocab) - len(medical_words)}\n\n")
            
            # Category breakdown
            f.write("Medical Category Breakdown:\n")
            for category, terms in self.medical_terms.items():
                category_words = [word for word in vocab if self.get_medical_category(word) == category]
                f.write(f"  - {category.replace('_', ' ').title()}: {len(category_words)} words\n")
                if category_words:
                    f.write(f"    Examples: {', '.join(category_words[:5])}\n")
            
            f.write(f"\nMedical words with affixes: {len([word for word in medical_words if self.get_medical_category(word) == 'medical_affixes'])}\n")
            
            if medical_words:
                f.write(f"\nAverage medical word length: {np.mean([len(word) for word in medical_words]):.2f} characters\n")
                f.write(f"Longest medical word: {max(medical_words, key=len)} ({len(max(medical_words, key=len))} chars)\n")
                f.write(f"Shortest medical word: {min(medical_words, key=len)} ({len(min(medical_words, key=len))} chars)\n")
        
        print(f"Results saved to: {output_dir}")
        print(f"  - all_words_coordinates.csv: {len(df_all)} word coordinates")
        print(f"  - medical_words_coordinates.csv: {len(df_medical)} medical word coordinates")
        print(f"  - cluster_analysis_report.txt: cluster analysis report")
        print(f"  - medical_vocabulary_stats.txt: medical vocabulary statistics report")
        return df_all

    def run_complete_pipeline(self, n_samples=1000, output_dir='./medical_visualization_results'):
        """Execute the full medical word visualization pipeline"""
        print("Starting enhanced medical word visualization pipeline...")
        print("=" * 60)
        output_dir = Path(output_dir)
        output_dir.mkdir(exist_ok=True)
        random.seed(42)
        np.random.seed(42)
        
        if not self.load_word_vectors():
            print("Data loading failed, exiting...")
            return False
        
        # Show medical vocabulary statistics
        total_medical = sum(1 for word in self.vocab if self.is_medical_word(word))
        print(f"Medical vocabulary in dataset: {total_medical}/{len(self.vocab)} ({total_medical/len(self.vocab)*100:.2f}%)")
        
        # Show category distribution
        category_distribution = {}
        for word in self.vocab:
            category = self.get_medical_category(word)
            if category:
                category_distribution[category] = category_distribution.get(category, 0) + 1
        
        print("Medical category distribution:")
        for category, count in sorted(category_distribution.items(), key=lambda x: x[1], reverse=True):
            print(f"  - {category.replace('_', ' ').title()}: {count} words")
        
        vectors, vocab, indices = self.random_sampling(n_samples)
        cluster_labels, cluster_info = self.run_clustering(vectors, vocab, n_clusters=8)
        
        embeddings_2d = self.run_tsne(vectors, dim=2, perplexity=30)
        fig_2d = self.create_2d_plot(
            embeddings_2d, vocab, cluster_labels,
            save_path=output_dir / 'medical_tsne_2d.png'
        )
        
        embeddings_3d = self.run_tsne(vectors, dim=3, perplexity=30)
        fig_3d = self.create_3d_plot(
            embeddings_3d, vocab, cluster_labels,
            save_path=output_dir / 'medical_tsne_3d.png'
        )
        
        df_all = self.save_detailed_results(
            vocab, embeddings_2d, embeddings_3d, cluster_labels, cluster_info, output_dir
        )
        
        self.generate_summary_report(vocab, cluster_info, output_dir, category_distribution)
        
        print("\n" + "=" * 60)
        print("Enhanced medical word visualization complete!")
        print(f"Output directory: {output_dir}")
        print(f"Visualized words: {len(vocab)} words")
        medical_in_sample = sum(1 for word in vocab if self.is_medical_word(word))
        print(f"Medical words in visualization: {medical_in_sample}/{len(vocab)} ({medical_in_sample/len(vocab)*100:.1f}%)")
        print("Generated files:")
        print("  medical_tsne_2d.png - 2D medical word visualization with category colors")
        print("  medical_tsne_3d.png - 3D medical word visualization with category colors")
        print("  all_words_coordinates.csv - all word coordinates with categories")
        print("  medical_words_coordinates.csv - medical word coordinates")
        print("  cluster_analysis_report.txt - detailed cluster analysis")
        print("  medical_vocabulary_stats.txt - comprehensive medical vocabulary statistics")
        print("  visualization_summary.txt - visualization summary report")
        return True

    def generate_summary_report(self, vocab, cluster_info, output_dir, category_distribution):
        """Generate a comprehensive summary report of medical word visualizations"""
        # Count medical words in sample
        medical_count = sum(1 for word in vocab if self.is_medical_word(word))
        
        with open(output_dir / 'visualization_summary.txt', 'w', encoding='utf-8') as f:
            f.write("=== Enhanced Medical Word t-SNE Visualization Summary ===\n\n")
            f.write("Basic Statistics:\n")
            f.write(f"  - Total words visualized: {len(vocab)}\n")
            f.write(f"  - Medical words identified: {medical_count} ({medical_count/len(vocab)*100:.1f}%)\n")
            f.write(f"  - Number of clusters: {len(cluster_info)}\n")
            f.write("  - Sampling method: Random sampling with stopword filtering\n")
            f.write("  - Label selection: Medical-related words prioritized by category\n")
            f.write(f"  - Stopwords filtered: {len(self.stopwords)} common words excluded\n\n")
            
            f.write("Medical Word Categories:\n")
            sample_categories = {}
            for word in vocab:
                category = self.get_medical_category(word)
                if category:
                    sample_categories[category] = sample_categories.get(category, 0) + 1
            
            for category, count in sorted(sample_categories.items(), key=lambda x: x[1], reverse=True):
                percentage = count/len(vocab)*100
                f.write(f"  - {category.replace('_', ' ').title()}: {count} words ({percentage:.1f}%)\n")
            
            f.write("\nVisualization Features:\n")
            f.write("  - Category-based color coding: Each medical category has unique color\n")
            f.write("  - 2D plot: Medical words labeled with category-specific colored backgrounds\n")
            f.write("  - 3D plot: Medical words shown in category colors for emphasis\n")
            f.write("  - Consistent coloring: Same category = same color across all plots\n")
            f.write("  - Enhanced filtering: Stopwords and non-alphanumeric words excluded\n\n")
            
            f.write("Cluster Analysis:\n")
            for i, info in enumerate(cluster_info):
                cluster_words = [vocab[j] for j in range(len(vocab)) if j < len(vocab)]
                medical_in_cluster = [word for word in info['sample_words'] if self.is_medical_word(word)]
                f.write(f"  Cluster {i+1}: {info['total_words']} words")
                if medical_in_cluster:
                    f.write(f" (medical: {', '.join(medical_in_cluster[:3])})")
                f.write("\n")
            
            f.write("\nTechnical Details:\n")
            f.write("  - Dimensionality reduction: t-SNE with PCA preprocessing\n")
            f.write("  - Clustering algorithm: K-means (k=8)\n")
            f.write("  - Feature scaling: StandardScaler normalization\n")
            f.write("  - Medical detection: Comprehensive keyword + affix pattern matching\n")
            f.write(f"  - Color palette: {len(self.medical_colors)} distinct category colors\n\n")
            
            f.write("Key Insights:\n")
            f.write("  1. Enhanced medical term coverage with comprehensive vocabulary\n")
            f.write("  2. Category-based visualization reveals semantic groupings\n")
            f.write("  3. Stopword filtering improves focus on meaningful terms\n")
            f.write("  4. Color coding enables quick identification of medical domains\n")
            f.write("  5. Affix pattern matching captures morphological medical terms\n\n")
            
            f.write("Medical Categories Explained:\n")
            for category, color in self.medical_colors.items():
                f.write(f"  - {category.replace('_', ' ').title()} (Color: {color}):\n")
                if category in self.medical_terms:
                    sample_terms = list(self.medical_terms[category])[:5]
                    f.write(f"    Examples: {', '.join(sample_terms)}\n")

# Main function
def main():
    """Main - Enhanced Medical word visualization with comprehensive filtering and categorization"""
    words_vectors_path = os.path.join(OUTPUT_DIR, 'skipgram', 'embeddings.txt')
    print("Enhanced Medical Word t-SNE Visualization System")
    print("=" * 50)
    print(f"Input file: {words_vectors_path}")
    print("New features:")
    print("  - Comprehensive stopword filtering")
    print("  - Expanded medical vocabulary (500+ terms)")
    print("  - Category-based color coding")
    print("  - Enhanced medical term detection")
    
    visualizer = MedicalWordVisualizer(words_vectors_path)
    success = visualizer.run_complete_pipeline(
        n_samples=1000,
        output_dir=os.path.join(OUTPUT_DIR, 'medical_visualization_results')
    )
    
    if success:
        print("\nEnhanced medical word visualization task completed!")
        print("Generated high-quality 2D and 3D plots with category-based coloring")
        print("Used random sampling of 1000 words with stopword filtering")
        print("Prioritized medical words with comprehensive categorization")
        print("Applied consistent color coding for medical categories")
        print("Provided detailed medical vocabulary analysis and category statistics")
    else:
        print("\nVisualization task failed")

if __name__ == "__main__":
    main()

### 4.3 Co-occurrence with "COVID" and "coronavirus"

Which entities appear within 5 words of "covid" and "coronavirus" in the tokenized corpus?
The results are shown as a network, a heatmap and frequency charts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict, Counter
import re
from itertools import combinations
from datetime import datetime
import os
from functools import lru_cache


class EnhancedCOVID19CooccurrenceAnalyzer:
    """Enhanced analyzer for high-frequency COVID-19 entity cooccurrences."""
    
    def __init__(self, input_file, target_words=None, window_size=5):
        self.input_file = input_file
        self.target_words = target_words or ["covid", "coronavirus"]
        self.window_size = window_size
        self.cooccurrence_matrix = defaultdict(Counter)
        self.entity_frequencies = Counter()
        self.documents = []
        
        # Predefined categories for entities
        self.entity_categories = {
            'symptom': ['fever', 'cough', 'pneumonia', 'breathing', 'fatigue', 'headache', 'loss', 'taste', 'smell'],
            'treatment': ['vaccine', 'drug', 'therapy', 'treatment', 'medicine', 'antiviral', 'hydroxychloroquine'],
            'anatomy': ['lung', 'respiratory', 'airway', 'blood', 'tissue', 'organ', 'heart', 'kidney'],
            'pathogen': ['virus', 'bacteria', 'infection', 'pathogen', 'microbe', 'strain', 'variant'],
            'medical': ['hospital', 'icu', 'ventilator', 'oxygen', 'patient', 'doctor', 'nurse', 'test', 'diagnosis']
        }
    
    def load_documents(self):
        """Load tokenized documents from the input file."""
        print("Loading tokenized documents...")
        try:
            with open(self.input_file, 'r', encoding='utf-8') as f:
                lines = f.read().splitlines()
            # Split each line into a list of tokens
            self.documents = [line.split() for line in lines if line.strip()]
            print(f"Successfully loaded {len(self.documents)} documents")
            return True
        except Exception as e:
            print(f"Error loading file: {e}")
            return False
    
    @lru_cache(maxsize=1000)
    def is_biomedical_entity(self, word):
        """Determine if a token is a biomedical entity (with caching)."""
        biomedical_patterns = [
            r'\b(protein|gene|virus|bacteria|cell|tissue|organ)\b',
            r'\b(symptom|disease|infection|syndrome|disorder)\b',
            r'\b(treatment|therapy|medicine|drug|vaccine)\b',
            r'\b(hospital|clinic|patient|doctor|nurse)\b',
            r'\b(test|diagnosis|screening|prevention)\b',
            r'\b\w+ase\b', r'\b\w+in\b', r'\b\w+itis\b',
        ]
        lower = word.lower()
        stop_words = {'the', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by'}
        
        if lower in stop_words or len(word) < 3:
            return False
        
        # Match against biomedical regex patterns
        for pattern in biomedical_patterns:
            if re.search(pattern, lower):
                return True
        
        # Additional medical term lookup
        medical_terms = {
            'pneumonia', 'fever', 'cough', 'breathing', 'lung', 'respiratory',
            'immune', 'antibody', 'antigen', 'mortality', 'morbidity',
            'epidemiology', 'pandemic', 'epidemic', 'transmission', 'spread',
            'quarantine', 'isolation', 'icu', 'ventilator', 'oxygen'
        }
        return lower in medical_terms
    
    def categorize_entity(self, entity):
        """Assign an entity to one of the predefined categories."""
        lower = entity.lower()
        for category, keywords in self.entity_categories.items():
            if any(keyword in lower for keyword in keywords):
                return category
        return 'other'
    
    def extract_cooccurrences(self):
        """Extract cooccurrence relationships around target words."""
        print("Extracting cooccurrences...")
        total = 0
        docs_with_target = 0
        
        for idx, doc in enumerate(self.documents):
            if idx % 1000 == 0:
                print(f"Progress: {idx}/{len(self.documents)} documents")
            lower_doc = [w.lower() for w in doc]
            positions = [i for i, w in enumerate(lower_doc)
                         if any(target in w for target in self.target_words)]
            if not positions:
                continue
            docs_with_target += 1
            
            for pos in positions:
                start = max(0, pos - self.window_size)
                end = min(len(doc), pos + self.window_size + 1)
                window = doc[start:end]
                
                for w in window:
                    if w.lower() != doc[pos].lower() and self.is_biomedical_entity(w):
                        tgt = doc[pos].lower()
                        ent = w.lower()
                        self.cooccurrence_matrix[tgt][ent] += 1
                        self.entity_frequencies[ent] += 1
                        total += 1
        
        print(f"Extraction complete! Documents with target: {docs_with_target}")
        print(f"Total cooccurrences found: {total}")
        print(f"Unique entities identified: {len(self.entity_frequencies)}")
    
    def get_top_cooccurrent_entities(self, top_n=50):
        """Return the top_n most frequent cooccurring entities across all targets."""
        combined = Counter()
        for tgt in self.target_words:
            combined.update(self.cooccurrence_matrix.get(tgt, {}))
        return combined.most_common(top_n)
    
    def create_enhanced_network_visualization(self, output_file="enhanced_network.png", top_n=30):
        """Generate an enhanced dashboard of cooccurrence visualizations."""
        print("Generating enhanced network visualization...")
        fig = plt.figure(figsize=(20, 12))
        gs = fig.add_gridspec(
            2, 3, 
            height_ratios=[3, 1], 
            width_ratios=[2, 1, 1],
            hspace=0.3, 
            wspace=0.3)
        
        ax_net = fig.add_subplot(gs[0, :2])
        self._plot_main_network(ax_net, top_n)
        
        ax_freq = fig.add_subplot(gs[0, 2])
        self._plot_frequency_distribution(ax_freq, top_n // 2)
        
        ax_heat = fig.add_subplot(gs[1, 0])
        self._plot_cooccurrence_heatmap(ax_heat, 15)
        
        ax_pie = fig.add_subplot(gs[1, 1])
        self._plot_category_distribution(ax_pie, top_n)
        
        ax_scatter = fig.add_subplot(gs[1, 2])
        self._plot_cooccurrence_strength_scatter(ax_scatter, top_n // 2)
        
        plt.suptitle("Enhanced COVID-19 Entity Cooccurrence Dashboard",
                     fontsize=20, fontweight='bold')
        plt.savefig(output_file, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Enhanced network visualization saved to: {output_file}")
    
    def _plot_main_network(self, ax, top_n):
        """Plot the main cooccurrence network graph."""
        top_entities = self.get_top_cooccurrent_entities(top_n)
        G = nx.Graph()
        
        # Add target nodes
        for tgt in self.target_words:
            G.add_node(tgt, node_type='target', category='COVID')
        
        max_count = max((cnt for _, cnt in top_entities), default=1)
        for ent, cnt in top_entities:
            cat = self.categorize_entity(ent)
            G.add_node(ent, node_type='entity', category=cat, frequency=cnt)
            for tgt in self.target_words:
                w = self.cooccurrence_matrix[tgt].get(ent, 0)
                if w > 0:
                    G.add_edge(tgt, ent, weight=w)
        
        pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
        
        # Define color mapping per category
        color_map = {
            'COVID': '#FF4444', 'symptom': '#66B2FF', 'treatment': '#66FF66',
            'anatomy': '#FFB366', 'pathogen': '#FF66FF', 'medical': '#FFFF66',
            'other': '#CCCCCC'
        }
        
        # Draw nodes by category
        for cat, color in color_map.items():
            nodes = [n for n, d in G.nodes(data=True) if d.get('category') == cat]
            if not nodes:
                continue
            if cat == 'COVID':
                nx.draw_networkx_nodes(G, pos, nodelist=nodes, node_color=color,
                                       node_size=1500, alpha=0.9, ax=ax)
            else:
                sizes = [min(1000, G.nodes[n]['frequency'] * 15) for n in nodes]
                nx.draw_networkx_nodes(G, pos, nodelist=nodes, node_color=color,
                                       node_size=sizes, alpha=0.7, ax=ax)
        
        # Draw edges with widths proportional to weight
        edges = G.edges(data=True)
        if edges:
            weights = [d['weight'] for _, _, d in edges]
            max_w = max(weights)
            widths = [max(0.5, w / max_w * 4) for w in weights]
            nx.draw_networkx_edges(G, pos, width=widths, alpha=0.5, edge_color='gray', ax=ax)
        
        # Label high-frequency entities
        freqs = [G.nodes[n].get('frequency', 0) for n in G.nodes()]
        threshold = np.percentile(freqs, 60) if freqs else 0
        labels = {}
        for n in G.nodes():
            data = G.nodes[n]
            if data.get('node_type') == 'target':
                labels[n] = n.upper()
            elif data.get('frequency', 0) >= threshold:
                labels[n] = (n[:10] + '...') if len(n) > 10 else n
        nx.draw_networkx_labels(G, pos, labels, font_size=9, ax=ax)
        
        # Legend for categories
        legend_elems = [
            plt.Line2D([0], [0], marker='o', color='w',
                       markerfacecolor=color, markersize=10, label=cat.title())
            for cat, color in color_map.items() if cat != 'other'
        ]
        ax.legend(handles=legend_elems, loc='upper left', bbox_to_anchor=(0, 1))
        
        ax.set_title("Entity Cooccurrence Network", fontsize=14, fontweight='bold')
        ax.axis('off')
    
    def _plot_frequency_distribution(self, ax, top_n):
        """Plot a horizontal bar chart of top entity frequencies."""
        top_entities = self.get_top_cooccurrent_entities(top_n)
        labels = [e[:8] + '...' if len(e) > 8 else e for e, _ in top_entities]
        counts = [cnt for _, cnt in top_entities]
        bars = ax.barh(range(len(labels)), counts, alpha=0.8)
        
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=8)
        ax.set_xlabel('Cooccurrence Frequency', fontsize=10)
        ax.set_title('Top Entity Frequencies', fontsize=12, fontweight='bold')
        
        for i, (bar, cnt) in enumerate(zip(bars, counts)):
            ax.text(cnt + max(counts)*0.02, i, str(cnt),
                    va='center', fontsize=8, fontweight='bold')
    
    def _plot_cooccurrence_heatmap(self, ax, top_n):
        """Plot a heatmap of normalized cooccurrence strengths."""
        top_entities = self.get_top_cooccurrent_entities(top_n)
        entities = [ent for ent, _ in top_entities]
        
        matrix = []
        for tgt in self.target_words:
            row = []
            for ent in entities:
                raw = self.cooccurrence_matrix[tgt].get(ent, 0)
                norm = raw / max(1, self.entity_frequencies[ent])
                row.append(norm)
            matrix.append(row)
        
        im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto')
        ax.set_xticks(range(len(entities)))
        ax.set_yticks(range(len(self.target_words)))
        ax.set_xticklabels([e[:6] + '...' if len(e) > 6 else e for e in entities],
                           rotation=45, ha='right', fontsize=8)
        ax.set_yticklabels([w.upper() for w in self.target_words], fontsize=10)
        ax.set_title('Normalized Cooccurrence Heatmap', fontsize=12, fontweight='bold')
        plt.colorbar(im, ax=ax, shrink=0.8, label='Normalized Strength')
    
    def _plot_category_distribution(self, ax, top_n):
        """Plot a pie chart of entity category distribution."""
        top_entities = self.get_top_cooccurrent_entities(top_n)
        category_counts = Counter(self.categorize_entity(e) for e, _ in top_entities)
        
        cats = list(category_counts.keys())
        vals = list(category_counts.values())
        wedges, texts, autotexts = ax.pie(
            vals, labels=cats, autopct='%1.1f%%', startangle=90
        )
        for txt in autotexts:
            txt.set_color('white')
            txt.set_fontweight('bold')
        ax.set_title('Entity Category Distribution', fontsize=12, fontweight='bold')
    
    def _plot_cooccurrence_strength_scatter(self, ax, top_n):
        """Plot a scatter of raw vs. normalized cooccurrence strengths."""
        top_entities = self.get_top_cooccurrent_entities(top_n)
        freqs, raws, norms = [], [], []
        for ent, raw in top_entities:
            freq = self.entity_frequencies[ent]
            strength = raw / freq if freq > 0 else 0
            freqs.append(freq)
            raws.append(raw)
            norms.append(strength)
        
        sc = ax.scatter(freqs, raws, c=norms, s=80, alpha=0.7, cmap='plasma',
                        edgecolors='black', linewidth=0.5)
        ax.set_xlabel('Total Entity Frequency', fontsize=10)
        ax.set_ylabel('Cooccurrence Count', fontsize=10)
        ax.set_title('Cooccurrence Strength Analysis', fontsize=12, fontweight='bold')
        cbar = plt.colorbar(sc, ax=ax, shrink=0.8)
        cbar.set_label('Strength', rotation=270, labelpad=15, fontsize=9)
    
    def save_enhanced_results(self, output_dir, top_n=50):
        """Save enhanced results to CSV and return a DataFrame."""
        os.makedirs(output_dir, exist_ok=True)
        top_entities = self.get_top_cooccurrent_entities(top_n)
        rows = []
        for ent, total in top_entities:
            freq = self.entity_frequencies[ent]
            strength = total / freq if freq > 0 else 0
            row = {
                'Entity': ent,
                'Category': self.categorize_entity(ent),
                'Total_Cooccurrence': total,
                'Entity_Frequency': freq,
                'Cooccurrence_Strength': strength
            }
            for tgt in self.target_words:
                row[f'Cooccurrence_with_{tgt}'] = self.cooccurrence_matrix[tgt].get(ent, 0)
            rows.append(row)
        df = pd.DataFrame(rows)
        csv_path = os.path.join(output_dir, "enhanced_cooccurrence_results.csv")
        df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        return df
    
    def run_enhanced_analysis(self, output_dir=os.path.join(OUTPUT_DIR, 'covid19-cooccurrence')):
        """Run the full enhanced analysis pipeline."""
        print("Starting enhanced COVID-19 cooccurrence analysis...")
        if not self.load_documents():
            return
        self.extract_cooccurrences()
        if not self.entity_frequencies:
            print("No biomedical entities found. Please check the input data.")
            return
        viz_path = os.path.join(output_dir, "enhanced_cooccurrence_visualization.png")
        self.create_enhanced_network_visualization(viz_path)
        df = self.save_enhanced_results(output_dir)
        print("\nEnhanced analysis complete!")
        print(f"- Visualization file: {viz_path}")
        print(f"- Results file: {os.path.join(output_dir, 'enhanced_cooccurrence_results.csv')}")
        print("\nSummary statistics:")
        print(f"- Unique biomedical entities: {len(self.entity_frequencies)}")
        print("- Top 5 cooccurring entities:")
        for i, (ent, cnt) in enumerate(self.get_top_cooccurrent_entities(5), 1):
            cat = self.categorize_entity(ent)
            print(f"  {i}. {ent} ({cat}): {cnt} occurrences")

def main():
    """Main entry point."""
    input_file = os.path.join(OUTPUT_DIR, 'scispacy_tokenized_docs.txt')
    analyzer = EnhancedCOVID19CooccurrenceAnalyzer(
        input_file=input_file,
        target_words=["covid", "coronavirus"],
        window_size=5
    )
    analyzer.run_enhanced_analysis()

if __name__ == "__main__":
    main()

### 4.4 Semantic similarity

Which words have the closest vectors (cosine similarity, N-gram vectors) to "covid" and
"coronavirus", once stopwords and non-medical terms are filtered out?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import networkx as nx
from datetime import datetime
import os
import re
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

class COVID19SemanticSimilarityAnalyzer:
    """COVID-19 Semantic Similarity Entity Analyzer"""
    
    def __init__(self, vectors_file, vocab_file, target_words=None):
        """
        Initialize the analyzer
        
        Args:
            vectors_file: Path to the word vectors (.npy) file
            vocab_file: Path to the vocabulary (.txt) file
            target_words: List of target words to analyze
        """
        self.vectors_file = vectors_file
        self.vocab_file = vocab_file
        self.target_words = target_words or ["covid", "coronavirus"]
        self.word_vectors = None
        self.vocab_list = []
        self.word_to_index = {}
        self.target_indices = {}
        self.similarity_results = {}
        
        # Configure font settings for plots
        
    def load_data(self):
        """Load word vectors and vocabulary"""
        print("Loading word vectors and vocabulary...")
        try:
            self.word_vectors = np.load(self.vectors_file)
            print(f"Word vectors shape: {self.word_vectors.shape}")
            with open(self.vocab_file, 'r', encoding='utf-8') as f:
                self.vocab_list = [line.strip() for line in f if line.strip()]
            print(f"Vocabulary size: {len(self.vocab_list)}")
            self.word_to_index = {w: i for i, w in enumerate(self.vocab_list)}
            if len(self.vocab_list) != self.word_vectors.shape[0]:
                print(f"Warning: Vocabulary size ({len(self.vocab_list)}) ≠ vectors rows ({self.word_vectors.shape[0]})")
                min_size = min(len(self.vocab_list), self.word_vectors.shape[0])
                self.vocab_list = self.vocab_list[:min_size]
                self.word_vectors = self.word_vectors[:min_size]
                self.word_to_index = {w: i for i, w in enumerate(self.vocab_list)}
            self.find_target_words()
            return True
        except FileNotFoundError as e:
            print(f"File not found: {e}")
            return False
        except Exception as e:
            print(f"Error loading data: {e}")
            return False
    
    def find_target_words(self):
        """Locate indices of target words in the vocabulary"""
        self.target_indices = {}
        found = []
        for t in self.target_words:
            if t in self.word_to_index:
                self.target_indices[t] = self.word_to_index[t]
                found.append(t)
            else:
                matches = [w for w in self.vocab_list if t.lower() in w.lower()]
                if matches:
                    best = min(matches, key=len)
                    self.target_indices[t] = self.word_to_index[best]
                    found.append(f"{t}→{best}")
                else:
                    print(f"Warning: '{t}' not found")
        print(f"Found target words: {found}")
    
    def is_biomedical_entity(self, word):
        """
        Determine if a token is a biomedical entity via patterns and whitelist.
        """
        w = word.lower()
        stop_words = {
            'the','and','or','but','in','on','at','to','for','of','with','by',
            'a','an','is','are','was','were','be','been','have','has','had',
            'do','does','did','will','would','could','should','may','might',
            'this','that','these','those','i','you','he','she','it','we','they'
        }
        if w in stop_words or len(word) < 3 or word.isdigit():
            return False
        
        patterns = [
            r'\b(protein|gene|virus|bacteria|cell|tissue|organ|dna|rna)\b',
            r'\b(symptom|disease|infection|syndrome|disorder|condition)\b',
            r'\b(treatment|therapy|medicine|drug|vaccine|medication)\b',
            r'\b(hospital|clinic|patient|doctor|nurse|physician)\b',
            r'\b(test|diagnosis|screening|prevention|immunization)\b',
            r'\b(respiratory|cardiac|pulmonary|neurological|hepatic)\b',
            r'\b(acute|chronic|severe|mild|moderate|clinical)\b',
            r'\b\w*ase\b', r'\b\w*in\b',
            r'\b\w*itis\b', r'\b\w*osis\b',
            r'\b\w*emia\b', r'\b\w*pathy\b',
        ]
        for pat in patterns:
            if re.search(pat, w):
                return True
        
        medical_terms = {
            'coronavirus','covid','sars','mers','pandemic','epidemic','outbreak',
            'quarantine','isolation','lockdown','social_distancing',
            'fever','cough','fatigue','headache','nausea','vomiting','diarrhea',
            'dyspnea','shortness_of_breath','chest_pain','sore_throat',
            'lung','heart','liver','kidney','brain','blood','plasma',
            'respiratory_system','immune_system','cardiovascular',
            'ventilator','oxygen','icu','intensive_care','intubation',
            'remdesivir','dexamethasone','hydroxychloroquine','convalescent_plasma',
            'pcr','antigen_test','antibody','serology','ct_scan','x_ray',
            'biomarker','laboratory','specimen','transmission','incubation',
            'mortality','morbidity','prevalence','incidence','case_fatality_rate',
            'reproduction_number','immunity','antigen','immunoglobulin',
            'cytokine','inflammation','immune_response','vaccination','immunization'
        }
        return w in medical_terms
    
    def calculate_similarities(self, top_n=100):
        """Compute cosine similarities between each target and all vocabulary"""
        print("Calculating semantic similarities...")
        if not self.target_indices:
            print("Error: No targets")
            return
        for tgt, idx in self.target_indices.items():
            print(f"Processing '{tgt}'...")
            vec = self.word_vectors[idx].reshape(1, -1)
            sims = cosine_similarity(vec, self.word_vectors)[0]
            order = np.argsort(sims)[::-1]
            bio = []
            for i in order:
                if i == idx: continue
                w = self.vocab_list[i]
                if self.is_biomedical_entity(w):
                    bio.append((w, sims[i], i))
                if len(bio) >= top_n:
                    break
            self.similarity_results[tgt] = bio
            print(f"→ {len(bio)} biomedical entities found for '{tgt}'")
    
    def save_similarity_csv(self, output_file="covid19_semantic_similarity.csv", top_n=50):
        """Export both detailed and aggregated CSV reports"""
        print("Saving CSV reports...")
        rows = []
        for tgt, sims in self.similarity_results.items():
            for rank, (w, sim, i) in enumerate(sims[:top_n], start=1):
                rows.append({
                    'Target_Word': tgt,
                    'Rank': rank,
                    'Similar_Entity': w,
                    'Cosine_Similarity': sim,
                    'Word_Index': i
                })
        agg_map = defaultdict(list)
        for r in rows:
            agg_map[r['Similar_Entity']].append(r['Cosine_Similarity'])
        agg = [{
            'Entity': w,
            'Average_Similarity': np.mean(vals),
            'Max_Similarity': np.max(vals),
            'Appearance_Count': len(vals)
        } for w, vals in agg_map.items()]
        agg.sort(key=lambda x: x['Average_Similarity'], reverse=True)
        
        df_det = pd.DataFrame(rows)
        det_file = output_file.replace('.csv', '_detailed.csv')
        df_det.to_csv(det_file, index=False, encoding='utf-8-sig')
        
        df_agg = pd.DataFrame(agg[:top_n])
        df_agg.to_csv(output_file, index=False, encoding='utf-8-sig')
        
        print(f"→ Aggregated: {output_file}\n→ Detailed: {det_file}")
        return df_agg, df_det
    
    def create_similarity_visualization(self, output_file="covid19_semantic_similarity.png", top_n=30):
        """Generate a 2×2 figure: heatmap, t-SNE, network, and histogram"""
        print("Creating visualization figure...")
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 16))
        self._create_similarity_heatmap(ax1, top_n)
        self._create_tsne_plot(ax2, top_n)
        self._create_network_plot(ax3, top_n)
        self._create_similarity_distribution(ax4, top_n)
        plt.tight_layout()
        plt.savefig(output_file, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"→ Saved figure: {output_file}")
    
    def _create_similarity_heatmap(self, ax, top_n):
        """Draw a heatmap of similarity scores for top entities"""
        # Gather a limited set of entities across all targets
        all_entities = set()
        for sims in self.similarity_results.values():
            for w, _, _ in sims[:top_n // 2]:
                all_entities.add(w)
        entities = list(all_entities)[:20]
        
        matrix = np.zeros((len(self.target_indices), len(entities)))
        targets = list(self.target_indices.keys())
        for i, tgt in enumerate(targets):
            sim_map = {w: s for w, s, _ in self.similarity_results[tgt]}
            for j, ent in enumerate(entities):
                matrix[i, j] = sim_map.get(ent, 0)
        
        sns.heatmap(matrix,
                    xticklabels=[e[:15] for e in entities],
                    yticklabels=targets,
                    annot=True, fmt='.3f', cmap='YlOrRd',
                    ax=ax)
        ax.set_title('Heatmap of Semantic Similarities')
        ax.set_xlabel('Entities')
        ax.set_ylabel('Target Words')
    
    def _create_tsne_plot(self, ax, top_n):
        """Embed words via t-SNE and plot in 2D scatter"""
        indices = []
        labels = []
        colors = []
        cmap = {'covid': 'red', 'coronavirus': 'blue'}
        
        # Add target points
        for tgt, idx in self.target_indices.items():
            indices.append(idx)
            labels.append(tgt.upper())
            colors.append(cmap.get(tgt, 'red'))
        
        # Add top entities
        seen = set()
        for sims in self.similarity_results.values():
            for w, _, idx in sims[:top_n // 4]:
                if w not in seen:
                    indices.append(idx)
                    labels.append(w[:10])
                    colors.append('lightblue')
                    seen.add(w)
        
        if len(indices) < 5:
            ax.text(0.5, 0.5, 'Insufficient data for t-SNE', ha='center', va='center')
            return
        
        vecs = self.word_vectors[indices]
        tsne = TSNE(n_components=2, random_state=42,
                    perplexity=min(30, len(indices)//2))
        embed = tsne.fit_transform(vecs)
        
        for i, (x, y) in enumerate(embed):
            ax.scatter(x, y, c=colors[i], s=100 if i < len(self.target_indices) else 50,
                       alpha=0.7, edgecolors='black' if i < len(self.target_indices) else None)
            ax.annotate(labels[i], (x, y), xytext=(5, 5), textcoords='offset points', fontsize=8)
        
        ax.set_title('t-SNE Embedding of Entities')
        ax.set_xlabel('Dimension 1')
        ax.set_ylabel('Dimension 2')
    
    def _create_network_plot(self, ax, top_n):
        """Build a network graph of targets linked to similar entities"""
        G = nx.Graph()
        for tgt in self.target_indices:
            G.add_node(tgt, type='target', size=1000)
        for tgt, sims in self.similarity_results.items():
            for w, sim, _ in sims[:top_n // 3]:
                G.add_node(w, type='entity', size=sim * 500)
                G.add_edge(tgt, w, weight=sim)
        
        pos = nx.spring_layout(G, k=2, iterations=50)
        
        tgt_nodes = [n for n, d in G.nodes(data=True) if d['type']=='target']
        ent_nodes = [n for n in G if n not in tgt_nodes]
        
        nx.draw_networkx_nodes(G, pos, nodelist=tgt_nodes,
                               node_color='red', node_size=1000, alpha=0.8, ax=ax)
        if ent_nodes:
            sizes = [G.nodes[n]['size'] for n in ent_nodes]
            nx.draw_networkx_nodes(G, pos, nodelist=ent_nodes,
                                   node_color='lightblue', node_size=sizes, alpha=0.6, ax=ax)
        
        edges = G.edges()
        if edges:
            weights = [G[u][v]['weight'] for u,v in edges]
            max_w = max(weights)
            widths = [w / max_w * 3 for w in weights]
            nx.draw_networkx_edges(G, pos, width=widths, alpha=0.5, ax=ax)
        
        nx.draw_networkx_labels(G, pos,
                                labels={n: n[:10] for n in G.nodes()},
                                font_size=8, ax=ax)
        ax.set_title('Similarity Network Graph')
        ax.axis('off')
    
    def _create_similarity_distribution(self, ax, top_n):
        """Plot a histogram of all similarity scores"""
        scores = []
        for sims in self.similarity_results.values():
            scores.extend([s for _, s, _ in sims[:top_n]])
        if not scores:
            ax.text(0.5, 0.5, 'No similarity data', ha='center', va='center')
            return
        
        ax.hist(scores, bins=20, alpha=0.7, edgecolor='black')
        ax.set_title('Distribution of Cosine Similarities')
        ax.set_xlabel('Cosine Similarity')
        ax.set_ylabel('Frequency')
        
        mean, med = np.mean(scores), np.median(scores)
        ax.axvline(mean, linestyle='--', label=f'Mean: {mean:.3f}')
        ax.axvline(med, linestyle='--', label=f'Median: {med:.3f}')
        ax.legend()
    
    def generate_analysis_report(self, output_file="covid19_semantic_analysis_report.txt", top_n=50):
        """Generate a textual analysis report to a .txt file"""
        print("Generating analysis report...")
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write("COVID-19 Semantic Similarity Analysis Report\n")
            f.write("="*50 + "\n\n")
            f.write(f"Analysis Timestamp: {datetime.now():%Y-%m-%d %H:%M:%S}\n")
            f.write(f"Word Vectors File: {self.vectors_file}\n")
            f.write(f"Vocabulary File: {self.vocab_file}\n")
            f.write(f"Target Words: {', '.join(self.target_words)}\n\n")
            
            f.write("Data Statistics:\n")
            f.write("-"*20 + "\n")
            f.write(f"Vector Matrix Shape: {self.word_vectors.shape}\n")
            f.write(f"Vocabulary Size: {len(self.vocab_list)}\n")
            f.write(f"Targets Found: {len(self.target_indices)}\n\n")
            
            for tgt, sims in self.similarity_results.items():
                f.write(f"Top Entities for {tgt.upper()}:\n")
                f.write("-"*30 + "\n")
                f.write(f"{'Rank':<5}{'Entity':<25}{'Cosine Similarity':<18}\n")
                f.write("-"*48 + "\n")
                for rank, (w, s, _) in enumerate(sims[:top_n], 1):
                    f.write(f"{rank:<5}{w:<25}{s:<18.4f}\n")
                f.write("\n")
            
        
        print(f"Report saved: {output_file}")
    
    def run_analysis(self, output_dir=os.path.join(OUTPUT_DIR, "covid19_semantic_similarity")):
        """Run the entire pipeline: load, compute, save, visualize, and report."""
        print("Starting COVID-19 Semantic Similarity Analysis...")
        os.makedirs(output_dir, exist_ok=True)
        
        if not self.load_data():
            return
        
        self.calculate_similarities()
        if not self.similarity_results:
            print("No results: please check your inputs.")
            return
        
        csv_path = os.path.join(output_dir, "covid19_semantic_similarity.csv")
        viz_path = os.path.join(output_dir, "covid19_semantic_similarity.png")
        rpt_path = os.path.join(output_dir, "covid19_semantic_analysis_report.txt")
        
        self.save_similarity_csv(csv_path)
        self.create_similarity_visualization(viz_path)
        self.generate_analysis_report(rpt_path)
        
        print("\nAnalysis complete!\n"
              f"- CSV: {csv_path}\n"
              f"- Visualization: {viz_path}\n"
              f"- Report: {rpt_path}")

def main():
    """Main entry point"""
    vectors_file = os.path.join(OUTPUT_DIR, 'ngram_vectors.npy')
    vocab_file = os.path.join(OUTPUT_DIR, 'ngram_vocab.txt')
    target_words = ["covid", "coronavirus"]
    
    analyzer = COVID19SemanticSimilarityAnalyzer(
        vectors_file=vectors_file,
        vocab_file=vocab_file,
        target_words=target_words
    )
    analyzer.run_analysis()

if __name__ == "__main__":
    main()

## Takeaways

- **Tokenization matters in biomedicine.** `split()` and NLTK break medical terms apart;
  SciSpaCy keeps them whole; subword tokenizers never meet an unknown word but produce
  pieces that are hard to read.
- **Word vectors need data.** Trained on small samples, our vectors capture broad structure
  (numbers, languages, groups of medical terms) and some sensible neighbours of
  "coronavirus" (disease-2019, virus, syndrome, SARS, MERS), but the similarities stay low
  (mean cosine similarity 0.26) and some neighbours are noise.
- **Next steps:** train on the full corpus, evaluate the vectors on a biomedical similarity
  benchmark instead of by eye, and use contextual models such as BioBERT for downstream tasks.